In [ ]:
!pip -q uninstall -y torchao
!pip -q install peft imbalanced-learn tabulate scipy matplotlib jinja2
print("Cell 1 complete — now click: Runtime > Restart session")

Cell 1 complete — now click: Runtime > Restart session


In [ ]:
import os, json, time, random, re, gc, shutil, zipfile, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import f1_score, accuracy_score, confusion_matrix
import peft, transformers
from peft.import_utils import is_torchao_available
from transformers import AutoTokenizer, AutoModelForSequenceClassification, AutoModel, get_linear_schedule_with_warmup

warnings.filterwarnings("ignore")
transformers.logging.set_verbosity_error()

assert not is_torchao_available(), "torchao is still loaded in memory — click Runtime > Restart session first."
assert torch.cuda.is_available(), "No GPU detected — go to Runtime > Change runtime type > T4 GPU."
DEVICE = "cuda:0"

# ---------------- 1. Mount Google Drive & Unpack Checkpoint Zip ----------------
from google.colab import drive
drive.mount("/content/drive")

DRIVE_ROOT = Path("/content/drive/MyDrive")
BASE_DIR   = DRIVE_ROOT / "dcm_bench"
WORK       = BASE_DIR / "main"
DATA_DIR   = BASE_DIR / "data"
RESULTS    = WORK / "results.jsonl"
BACKUP_DIR = BASE_DIR / "backups"

if not RESULTS.exists():
    zips = sorted(DRIVE_ROOT.glob("dcm_bench*.zip"), key=lambda p: p.stat().st_mtime, reverse=True)
    assert zips, f"No dcm_bench*.zip found in {DRIVE_ROOT}. Confirm the zip is in My Drive."
    zip_file = zips[0]
    print(f"Unpacking {zip_file.name} into Google Drive...")
    with zipfile.ZipFile(zip_file, "r") as zf:
        members = [m for m in zf.namelist() if not m.startswith("__MACOSX")]
        has_top_folder = any(m.startswith("dcm_bench/") for m in members)
        extract_target = DRIVE_ROOT if has_top_folder else BASE_DIR
        extract_target.mkdir(parents=True, exist_ok=True)
        for m in members:
            zf.extract(m, extract_target)
    print("Unpack complete.")

assert RESULTS.exists(), f"Extraction finished but {RESULTS} was not found. Check zip structure."

for sub in ["preds", "emb", "tables", "figs"]:
    (WORK / sub).mkdir(parents=True, exist_ok=True)
BACKUP_DIR.mkdir(parents=True, exist_ok=True)

LR = json.loads((WORK / "lr_config.json").read_text(encoding="utf-8")) if (WORK / "lr_config.json").exists() else {"full": 5e-5, "lora": 5e-4, "head": 1e-3}

# ---------------- 2. Mandatory Per-Cell Backup Helper ----------------
def backup_now(tag: str):
    stamp = time.strftime("%Y%m%d_%H%M%S")
    if RESULTS.exists():
        shutil.copy2(RESULTS, BACKUP_DIR / f"results_{tag}_{stamp}.jsonl")

    zip_path = BACKUP_DIR / f"dcm_backup_{tag}.zip"
    with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
        for fname in ["results.jsonl", "results_pre_pooler_fix_backup.jsonl", "lr_config.json", "PLAN.md", "best_config.json"]:
            fp = WORK / fname
            if fp.exists():
                zf.write(fp, arcname=f"main/{fname}")
        for folder in ["preds", "tables", "figs"]:
            fdir = WORK / folder
            if fdir.exists():
                for fp in fdir.rglob("*"):
                    if fp.is_file():
                        zf.write(fp, arcname=str(fp.relative_to(BASE_DIR)))

    n_done = len({json.loads(l)["run_id"] for l in open(RESULTS, encoding="utf-8") if l.strip()})
    mb = zip_path.stat().st_size / (1024 * 1024)
    print(f"[BACKUP SAVED] {zip_path} ({mb:.2f} MB) | Unique completed runs: {n_done}")

# ---------------- 3. Experiment Config, Data & Noise/Transfer Sets ----------------
LANGS = ["tamil", "kannada"]
MODELS = {
    "xlmr":  "xlm-roberta-base",
    "muril": "google/muril-base-cased",
    "mbert": "bert-base-multilingual-cased",
}
TUNINGS = {
    "full":     {},
    "lora_r8":  {"r": 8,  "alpha": 16},
    "lora_r32": {"r": 32, "alpha": 64},
}
FT_IMBALANCE     = ["ce", "wce", "focal", "ros"]
FROZEN_IMBALANCE = ["frozen_ce", "frozen_wce", "frozen_ros", "frozen_smote"]
SEEDS = [13, 21, 42, 87, 101]
MAX_LEN, EPOCHS, BS = 64, 3, 32
ROS_FRAC, FOCAL_GAMMA, NOISE_SEED = 0.5, 2.0, 0

CLASSES = ["positive", "negative", "mixed_feelings", "unknown_state", "not_in_language"]
K = len(CLASSES)
C2I = {c: i for i, c in enumerate(CLASSES)}

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)

def norm_label(l):
    l = re.sub(r"[\s\-]+", "_", str(l).strip().lower()).strip("_")
    if l.startswith("not_"): return "not_in_language"
    if l in ("mixed_feelings", "mixed", "mixed_feeling"): return "mixed_feelings"
    if l in ("unknown_state", "unknown"): return "unknown_state"
    return l

def norm_text(t):
    return re.sub(r"\s+", " ", str(t).strip().lower())

def read_local(path):
    with open(path, encoding="utf-8", errors="replace") as fh:
        raw_lines = [ln.strip() for ln in fh if ln.strip()]
    best_rows = []
    for sep in ("\t", ";", ","):
        rows = []
        for ln in raw_lines:
            parts = [p.strip() for p in ln.split(sep)]
            while parts and parts[-1] == "": parts.pop()
            if len(parts) < 2: continue
            lbl = norm_label(parts[-1])
            if lbl not in CLASSES: continue
            txt_parts = parts[1:-1] if (len(parts) >= 3 and re.fullmatch(r"\d+", parts[0])) else parts[:-1]
            txt = sep.join(txt_parts).strip()
            if txt: rows.append((txt, lbl))
        if len(rows) > len(best_rows): best_rows = rows
    return pd.DataFrame(best_rows, columns=["text", "label"])

def fetch_dataset_if_missing():
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    if any(p.suffix.lower() in (".tsv", ".csv", ".txt") for p in DATA_DIR.rglob("*")):
        return
    import urllib.request
    zpath = DATA_DIR / "dcm.zip"
    url = "https://zenodo.org/records/4750858/files/DravidianCodeMix-2020.zip?download=1"
    print("Downloading Zenodo dataset fallback:", url)
    urllib.request.urlretrieve(url, zpath)
    with zipfile.ZipFile(zpath) as z: z.extractall(DATA_DIR)

def find_files(lang):
    fetch_dataset_if_missing()
    found = {}
    for p in sorted(DATA_DIR.rglob("*")):
        n = p.name.lower()
        if "__macosx" in [x.lower() for x in p.parts] or n.startswith("."): continue
        if p.suffix.lower() not in (".tsv", ".csv", ".txt") or lang not in n or "offens" in n or "without_label" in n: continue
        for split, keys in [("train", ["train"]), ("dev", ["dev", "val"]), ("test", ["test"])]:
            if any(k in n for k in keys):
                if split not in found or ("sentiment" in n and "sentiment" not in found[split].name.lower()):
                    found[split] = p
    return found

DATA = {}
for lang in LANGS:
    files = find_files(lang)
    splits = {s: read_local(p) for s, p in files.items()}
    tr, dv, te = splits["train"], splits["dev"], splits["test"]
    tr = tr.assign(_k=tr.text.map(norm_text)).drop_duplicates("_k")
    held = set(dv.text.map(norm_text)) | set(te.text.map(norm_text))
    tr = tr[~tr._k.isin(held)].drop(columns="_k").reset_index(drop=True)
    for d in (tr, dv, te): d["y"] = d.label.map(C2I)
    DATA[lang] = {"train": tr, "dev": dv, "test": te}
    print(f"[{lang}] train={len(tr)} | dev={len(dv)} | test={len(te)}")

VOWELS = set("aeiou")
TRANSLIT_RULES = [
    ("aa", "a"), ("a", "aa"), ("ee", "i"), ("oo", "u"), ("u", "oo"),
    ("th", "t"), ("zh", "l"), ("sh", "s"), ("w", "v"), ("v", "w"),
    ("ai", "ei"), ("ph", "f"),
]
def _is_ascii_word(w): return w.isascii() and any(ch.isalpha() for ch in w)
def n_elongate(t, rng, p=0.15):
    out = []
    for ch in t:
        out.append(ch)
        if ch.lower() in VOWELS and ch.isascii() and rng.random() < p:
            out.append(ch * int(rng.integers(1, 3)))
    return "".join(out)
def n_typo(t, rng, p=0.05):
    chars, i, out = list(t), 0, []
    while i < len(chars):
        ch = chars[i]
        if ch.isascii() and ch.isalpha() and rng.random() < p:
            op = rng.integers(0, 3)
            if op == 0: pass
            elif op == 1 and i + 1 < len(chars): out.extend([chars[i + 1], ch]); i += 1
            else: out.extend([ch, ch])
        else: out.append(ch)
        i += 1
    return "".join(out)
def n_translit(t, rng, p_word=0.5):
    words = t.split(" ")
    for j, w in enumerate(words):
        if _is_ascii_word(w) and rng.random() < p_word:
            lw = w.lower()
            cand = [(a, b) for a, b in TRANSLIT_RULES if a in lw]
            if cand:
                a, b = cand[int(rng.integers(0, len(cand)))]
                words[j] = lw.replace(a, b)
    return " ".join(words)
def n_combined(t, rng): return n_typo(n_elongate(n_translit(t, rng), rng), rng)

NOISES = {"noise_elong": n_elongate, "noise_typo": n_typo, "noise_translit": n_translit, "noise_combined": n_combined}
EVAL_SETS = {}
for lang in LANGS:
    te = DATA[lang]["test"]; texts, ys = te.text.tolist(), te.y.values
    sets = {"clean": (texts, ys)}
    for name, fn in NOISES.items():
        rng = np.random.default_rng(NOISE_SEED)
        sets[name] = ([fn(t, rng) for t in texts], ys)
    for other in LANGS:
        if other != lang:
            o = DATA[other]["test"]; sets[f"xl_{other}"] = (o.text.tolist(), o.y.values)
    EVAL_SETS[lang] = sets

# ---------------- 4. Patched Model, Training & Resumable Orchestrator ----------------
def fast_macro_f1(y, p, k=K):
    cm = np.bincount(y * k + p, minlength=k * k).reshape(k, k)
    tp = np.diag(cm).astype(float); fp = cm.sum(0) - tp; fn = cm.sum(1) - tp
    denom = 2 * tp + fp + fn
    return float(np.mean(np.where(denom > 0, 2 * tp / np.maximum(denom, 1), 0.0)))

def compute_metrics(y, p):
    return {
        "macro_f1":     float(f1_score(y, p, average="macro",    labels=list(range(K)), zero_division=0)),
        "weighted_f1":  float(f1_score(y, p, average="weighted", labels=list(range(K)), zero_division=0)),
        "acc":          float(accuracy_score(y, p)),
        "per_class_f1": f1_score(y, p, average=None, labels=list(range(K)), zero_division=0).tolist(),
    }

def class_weights(y):
    counts = np.bincount(y, minlength=K)
    return len(y) / (K * np.maximum(counts, 1))

def ros_indices(y, frac, rng):
    counts = np.bincount(y, minlength=K); target = int(frac * counts.max())
    idx = list(range(len(y)))
    for c in range(K):
        if 0 < counts[c] < target:
            idx.extend(rng.choice(np.where(y == c)[0], target - counts[c], replace=True).tolist())
    return np.array(idx)

def focal_loss(logits, y, gamma=FOCAL_GAMMA, weight=None):
    logp = F.log_softmax(logits.float(), -1)
    logpt = logp.gather(1, y[:, None]).squeeze(1); pt = logpt.exp()
    loss = -((1 - pt) ** gamma) * logpt
    if weight is not None: loss = loss * weight[y]
    return loss.mean()

_TOK, _FROZEN, _ENC_PARAMS = {}, {}, {}
def get_tok(model_key):
    if model_key not in _TOK: _TOK[model_key] = AutoTokenizer.from_pretrained(MODELS[model_key])
    return _TOK[model_key]

def lr_for(tuning, lr_override=None):
    return lr_override if lr_override is not None else (LR["full"] if tuning == "full" else LR["lora"])

def build_model(model_key, tuning):
    m = AutoModelForSequenceClassification.from_pretrained(
        MODELS[model_key], num_labels=K, id2label=dict(enumerate(CLASSES)), label2id=C2I
    )
    if tuning.startswith("lora"):
        from peft import LoraConfig, get_peft_model, TaskType
        p = TUNINGS[tuning]
        save_mods = ["classifier", "pooler"] if hasattr(m, "bert") else ["classifier"]
        cfg = LoraConfig(
            task_type=TaskType.SEQ_CLS, r=p["r"], lora_alpha=p["alpha"], lora_dropout=0.1,
            target_modules=["query", "value"], modules_to_save=save_mods,
        )
        m = get_peft_model(m, cfg)
    return m.to(DEVICE)

@torch.no_grad()
def predict(model, tok, texts, bs=128):
    model.eval()
    order = np.argsort([len(t) for t in texts]); out = np.zeros(len(texts), dtype=int)
    for i in range(0, len(texts), bs):
        ids = order[i:i + bs]
        enc = tok([texts[j] for j in ids], truncation=True, max_length=MAX_LEN, padding=True, return_tensors="pt").to(DEVICE)
        with torch.autocast("cuda", dtype=torch.float16):
            logits = model(**enc).logits
        out[ids] = logits.argmax(-1).cpu().numpy()
    return out

def train_finetune(lang, model_key, tuning, imb, seed, ros_frac=ROS_FRAC, lr_override=None):
    _FROZEN.clear(); gc.collect(); torch.cuda.empty_cache()
    set_seed(seed); rng = np.random.default_rng(seed)
    tr, dv = DATA[lang]["train"], DATA[lang]["dev"]
    texts, y = tr.text.tolist(), tr.y.values
    base_imb = imb.split("_f")[0]
    idx0 = ros_indices(y, ros_frac, rng) if base_imb == "ros" else np.arange(len(y))
    weight = torch.tensor(class_weights(y), dtype=torch.float, device=DEVICE) if base_imb == "wce" else None

    tok = get_tok(model_key); model = build_model(model_key, tuning)
    n_train = sum(p.numel() for p in model.parameters() if p.requires_grad)
    n_total = sum(p.numel() for p in model.parameters())
    used_lr = lr_for(tuning, lr_override)
    opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=used_lr, weight_decay=0.01)
    steps = EPOCHS * int(np.ceil(len(idx0) / BS))
    sched = get_linear_schedule_with_warmup(opt, int(0.06 * steps), steps)
    scaler = torch.amp.GradScaler("cuda")
    torch.cuda.reset_peak_memory_stats()

    train_time, best_f1, best_ep, best_state = 0.0, -1.0, -1, None
    for ep in range(EPOCHS):
        model.train(); perm = rng.permutation(idx0)
        torch.cuda.synchronize(); t0 = time.time()
        for i in range(0, len(perm), BS):
            b = perm[i:i + BS]
            enc = tok([texts[j] for j in b], truncation=True, max_length=MAX_LEN, padding=True, return_tensors="pt").to(DEVICE)
            yb = torch.tensor(y[b], dtype=torch.long, device=DEVICE)
            with torch.autocast("cuda", dtype=torch.float16):
                logits = model(**enc).logits
            if base_imb == "focal": loss = focal_loss(logits, yb)
            elif base_imb == "wce": loss = F.cross_entropy(logits.float(), yb, weight=weight)
            else: loss = F.cross_entropy(logits.float(), yb)
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt); scaler.update(); sched.step()
        torch.cuda.synchronize(); train_time += time.time() - t0
        dev_f1 = fast_macro_f1(dv.y.values, predict(model, tok, dv.text.tolist()))
        print(f"    ep{ep + 1} dev macro-F1 = {dev_f1:.4f}")
        if dev_f1 > best_f1:
            best_f1, best_ep = dev_f1, ep + 1
            best_state = {n: p.detach().cpu().clone() for n, p in model.named_parameters() if p.requires_grad}

    with torch.no_grad():
        for n, p in model.named_parameters():
            if n in best_state: p.copy_(best_state[n].to(p.device))
    res_sets, saved_preds = {}, {}
    for name, (tx, ys) in EVAL_SETS[lang].items():
        pr = predict(model, tok, tx); res_sets[name] = compute_metrics(ys, pr)
        if name in ("clean", "noise_combined"): saved_preds[name] = pr

    peak = torch.cuda.max_memory_allocated() / 1e9
    info = dict(
        trainable_params=n_train, total_params=n_total,
        trainable_pct=100 * n_train / n_total, train_time_s=train_time,
        peak_mem_gb=peak, best_epoch=best_ep, dev_macro_f1=best_f1,
        lr_used=used_lr, ros_frac=ros_frac
    )
    del model, opt, sched, best_state; gc.collect(); torch.cuda.empty_cache()
    return info, res_sets, saved_preds

@torch.no_grad()
def embed(model_key, texts, key, bs=128):
    f = WORK / "emb" / f"{model_key}_{key}.npy"
    if f.exists(): return np.load(f)
    if model_key not in _FROZEN:
        enc_m = AutoModel.from_pretrained(MODELS[model_key]).to(DEVICE).eval().half()
        _FROZEN[model_key] = enc_m
        _ENC_PARAMS[model_key] = sum(p.numel() for p in enc_m.parameters())
    enc_model, tok = _FROZEN[model_key], get_tok(model_key)
    order = np.argsort([len(t) for t in texts])
    out = np.zeros((len(texts), enc_model.config.hidden_size), dtype=np.float32)
    for i in range(0, len(texts), bs):
        ids = order[i:i + bs]
        enc = tok([texts[j] for j in ids], truncation=True, max_length=MAX_LEN, padding=True, return_tensors="pt").to(DEVICE)
        h = enc_model(**enc).last_hidden_state.float()
        m = enc["attention_mask"].unsqueeze(-1).float()
        out[ids] = ((h * m).sum(1) / m.sum(1).clamp(min=1e-9)).cpu().numpy()
    np.save(f, out)
    return out

def get_enc_param_count(model_key):
    if model_key not in _ENC_PARAMS:
        if model_key in _FROZEN: _ENC_PARAMS[model_key] = sum(p.numel() for p in _FROZEN[model_key].parameters())
        else:
            tmp = AutoModel.from_pretrained(MODELS[model_key])
            _ENC_PARAMS[model_key] = sum(p.numel() for p in tmp.parameters())
            del tmp; gc.collect()
    return _ENC_PARAMS[model_key]

def train_frozen(lang, model_key, imb, seed, ros_frac=ROS_FRAC):
    set_seed(seed); rng = np.random.default_rng(seed)
    base = imb.replace("frozen_", "").split("_f")[0]
    tr, dv = DATA[lang]["train"], DATA[lang]["dev"]
    Xtr = embed(model_key, tr.text.tolist(), f"{lang}_train"); ytr = tr.y.values
    Xdv = embed(model_key, dv.text.tolist(), f"{lang}_dev")
    mu, sd = Xtr.mean(0), Xtr.std(0) + 1e-6
    z = lambda X: (X - mu) / sd

    if base == "ros":
        idx = ros_indices(ytr, ros_frac, rng); X, y = Xtr[idx], ytr[idx]
    elif base == "smote":
        from imblearn.over_sampling import SMOTE
        counts = np.bincount(ytr, minlength=K); target = int(ros_frac * counts.max())
        strat = {c: target for c in range(K) if 1 < counts[c] < target}
        if strat:
            k = max(1, min(5, int(min(counts[c] for c in strat)) - 1))
            X_z, y = SMOTE(sampling_strategy=strat, k_neighbors=k, random_state=seed).fit_resample(z(Xtr), ytr)
            X = X_z * sd + mu
        else: X, y = Xtr, ytr
    else: X, y = Xtr, ytr

    head = nn.Sequential(nn.Linear(Xtr.shape[1], 256), nn.ReLU(), nn.Dropout(0.2), nn.Linear(256, K)).to(DEVICE)
    n_head = sum(p.numel() for p in head.parameters())
    n_enc = get_enc_param_count(model_key)
    opt = torch.optim.AdamW(head.parameters(), lr=LR["head"], weight_decay=0.01)
    Xt = torch.tensor(z(X), dtype=torch.float32, device=DEVICE); yt = torch.tensor(y, dtype=torch.long, device=DEVICE)
    Xd = torch.tensor(z(Xdv), dtype=torch.float32, device=DEVICE)
    w = torch.tensor(class_weights(ytr), dtype=torch.float, device=DEVICE) if base == "wce" else None

    torch.cuda.reset_peak_memory_stats()
    best_f1, best_state, best_ep, t0 = -1.0, None, -1, time.time()
    for ep in range(20):
        head.train(); perm = torch.randperm(len(Xt), device=DEVICE)
        for i in range(0, len(perm), 256):
            b = perm[i:i + 256]
            loss = F.cross_entropy(head(Xt[b]), yt[b], weight=w)
            opt.zero_grad(); loss.backward(); opt.step()
        head.eval()
        with torch.no_grad(): f = fast_macro_f1(dv.y.values, head(Xd).argmax(-1).cpu().numpy())
        if f > best_f1: best_f1, best_ep, best_state = f, ep + 1, {k_: v.clone() for k_, v in head.state_dict().items()}
    train_time = time.time() - t0

    head.load_state_dict(best_state); head.eval()
    res_sets, saved_preds = {}, {}
    for name, (tx, ys) in EVAL_SETS[lang].items():
        Xe = embed(model_key, tx, f"{lang}_{name}")
        with torch.no_grad(): pr = head(torch.tensor(z(Xe), dtype=torch.float32, device=DEVICE)).argmax(-1).cpu().numpy()
        res_sets[name] = compute_metrics(ys, pr)
        if name in ("clean", "noise_combined"): saved_preds[name] = pr

    peak = torch.cuda.max_memory_allocated() / 1e9
    info = dict(
        trainable_params=n_head, total_params=n_head + n_enc,
        trainable_pct=100 * n_head / (n_head + n_enc), train_time_s=train_time,
        peak_mem_gb=peak, best_epoch=best_ep, dev_macro_f1=best_f1,
        lr_used=LR["head"], ros_frac=ros_frac
    )
    del head, opt, Xt, yt, Xd, best_state; gc.collect()
    return info, res_sets, saved_preds

def run_id(r): return f"{r['lang']}|{r['model']}|{r['tuning']}|{r['imb']}|{r['seed']}"
def done_ids():
    if not RESULTS.exists(): return set()
    return {json.loads(l)["run_id"] for l in open(RESULTS, encoding="utf-8") if l.strip()}

def execute(runs):
    done = done_ids()
    todo = [r for r in runs if run_id(r) not in done]
    print(f"{len(runs)} runs requested | {len(runs) - len(todo)} already done | {len(todo)} remaining")
    t_stage = time.time()
    for n, r in enumerate(todo, 1):
        rid = run_id(r); frac = r.get("ros_frac", ROS_FRAC)
        lr_val = LR["head"] if r["imb"].startswith("frozen") else lr_for(r["tuning"])
        print(f"[{n}/{len(todo)}] {rid} (lr={lr_val}, ros_frac={frac})")
        try:
            if r["imb"].startswith("frozen"):
                info, sets, preds = train_frozen(r["lang"], r["model"], r["imb"], r["seed"], ros_frac=frac)
            else:
                info, sets, preds = train_finetune(r["lang"], r["model"], r["tuning"], r["imb"], r["seed"], ros_frac=frac)
        except torch.cuda.OutOfMemoryError:
            print("  OOM encountered — skipping run."); gc.collect(); torch.cuda.empty_cache(); continue

        row = dict(run_id=rid, lang=r["lang"], model=r["model"], tuning=r["tuning"], imb=r["imb"], seed=r["seed"], **info)
        for name, m in sets.items():
            row[f"{name}__macro_f1"] = m["macro_f1"]; row[f"{name}__acc"] = m["acc"]; row[f"{name}__weighted_f1"] = m["weighted_f1"]
        for c, f in zip(CLASSES, sets["clean"]["per_class_f1"]): row[f"clean__f1_{c}"] = f

        np.save(WORK / "preds" / (rid.replace("|", "__") + ".npy"), preds["clean"])
        if "noise_combined" in preds:
            np.save(WORK / "preds" / (rid.replace("|", "__") + "__noise_combined.npy"), preds["noise_combined"])

        with open(RESULTS, "a", encoding="utf-8") as fh:
            fh.write(json.dumps(row) + "\n")
            fh.flush()
            os.fsync(fh.fileno())
        gc.collect(); torch.cuda.empty_cache()
        print(f"  -> clean macro-F1 = {row['clean__macro_f1']:.4f} | dev = {info['dev_macro_f1']:.4f} | "
              f"trainable {info['trainable_pct']:.2f}% | {info['train_time_s']:.0f}s | elapsed {(time.time()-t_stage)/60:.1f}m")

print(f"\nGPU: {torch.cuda.get_device_name(0)} | Locked LR: {LR} | .npy files: {len(list((WORK/'preds').glob('*.npy')))}")
backup_now("cell2_restored")

MessageError: Error: credential propagation was unsuccessful

In [ ]:
import os, json, time, random, re, gc, shutil, zipfile, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import f1_score, accuracy_score, confusion_matrix
import peft, transformers
from peft.import_utils import is_torchao_available
from transformers import AutoTokenizer, AutoModelForSequenceClassification, AutoModel, get_linear_schedule_with_warmup

warnings.filterwarnings("ignore")
transformers.logging.set_verbosity_error()

assert not is_torchao_available(), "torchao is still loaded — click Runtime > Restart session first."
assert torch.cuda.is_available(), "No GPU detected — go to Runtime > Change runtime type > T4 GPU."
DEVICE = "cuda:0"

# ---------------- 1. Mount Google Drive (with fallback to /content if blocked) ----------------
USE_DRIVE = False
try:
    from google.colab import drive
    if not Path("/content/drive/MyDrive").exists():
        drive.mount("/content/drive", force_remount=True)
    if Path("/content/drive/MyDrive").exists():
        USE_DRIVE = True
except Exception as e:
    print("Drive mount skipped (using /content session storage):", e)

STORAGE_ROOT = Path("/content/drive/MyDrive") if USE_DRIVE else Path("/content")
BASE_DIR     = STORAGE_ROOT / "dcm_bench"
WORK         = BASE_DIR / "main"
DATA_DIR     = BASE_DIR / "data"
RESULTS      = WORK / "results.jsonl"
BACKUP_DIR   = BASE_DIR / "backups"

if not RESULTS.exists():
    zips = sorted(
        list(STORAGE_ROOT.glob("dcm_bench*.zip")) + list(Path("/content").glob("dcm_bench*.zip")),
        key=lambda p: p.stat().st_mtime,
        reverse=True,
    )
    assert zips, f"No dcm_bench*.zip found in {STORAGE_ROOT} or /content. Mount Drive or upload the zip in the Files sidebar."
    zip_file = zips[0]
    print(f"Unpacking {zip_file} into {STORAGE_ROOT}...")
    with zipfile.ZipFile(zip_file, "r") as zf:
        members = [m for m in zf.namelist() if not m.startswith("__MACOSX")]
        has_top_folder = any(m.startswith("dcm_bench/") for m in members)
        extract_target = STORAGE_ROOT if has_top_folder else BASE_DIR
        extract_target.mkdir(parents=True, exist_ok=True)
        for m in members:
            zf.extract(m, extract_target)
    print("Unpack complete.")

assert RESULTS.exists(), f"Extraction finished but {RESULTS} was not found."

for sub in ["preds", "emb", "tables", "figs"]:
    (WORK / sub).mkdir(parents=True, exist_ok=True)
BACKUP_DIR.mkdir(parents=True, exist_ok=True)

LR = json.loads((WORK / "lr_config.json").read_text(encoding="utf-8")) if (WORK / "lr_config.json").exists() else {"full": 5e-5, "lora": 5e-4, "head": 1e-3}

# ---------------- 2. Mandatory Per-Cell Backup Helper ----------------
def backup_now(tag: str):
    stamp = time.strftime("%Y%m%d_%H%M%S")
    if RESULTS.exists():
        shutil.copy2(RESULTS, BACKUP_DIR / f"results_{tag}_{stamp}.jsonl")

    zip_path = BACKUP_DIR / f"dcm_backup_{tag}.zip"
    with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
        for fname in ["results.jsonl", "results_pre_pooler_fix_backup.jsonl", "lr_config.json", "PLAN.md", "best_config.json"]:
            fp = WORK / fname
            if fp.exists():
                zf.write(fp, arcname=f"main/{fname}")
        for folder in ["preds", "tables", "figs"]:
            fdir = WORK / folder
            if fdir.exists():
                for fp in fdir.rglob("*"):
                    if fp.is_file():
                        zf.write(fp, arcname=str(fp.relative_to(BASE_DIR)))

    n_done = len({json.loads(l)["run_id"] for l in open(RESULTS, encoding="utf-8") if l.strip()})
    mb = zip_path.stat().st_size / (1024 * 1024)
    print(f"[BACKUP SAVED] {zip_path} ({mb:.2f} MB) | Unique completed runs: {n_done}")
    if not USE_DRIVE:
        from google.colab import files
        files.download(str(zip_path))

# ---------------- 3. Experiment Config, Data & Noise/Transfer Sets ----------------
LANGS = ["tamil", "kannada"]
MODELS = {
    "xlmr":  "xlm-roberta-base",
    "muril": "google/muril-base-cased",
    "mbert": "bert-base-multilingual-cased",
}
TUNINGS = {
    "full":     {},
    "lora_r8":  {"r": 8,  "alpha": 16},
    "lora_r32": {"r": 32, "alpha": 64},
}
FT_IMBALANCE     = ["ce", "wce", "focal", "ros"]
FROZEN_IMBALANCE = ["frozen_ce", "frozen_wce", "frozen_ros", "frozen_smote"]
SEEDS = [13, 21, 42, 87, 101]
MAX_LEN, EPOCHS, BS = 64, 3, 32
ROS_FRAC, FOCAL_GAMMA, NOISE_SEED = 0.5, 2.0, 0

CLASSES = ["positive", "negative", "mixed_feelings", "unknown_state", "not_in_language"]
K = len(CLASSES)
C2I = {c: i for i, c in enumerate(CLASSES)}

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)

def norm_label(l):
    l = re.sub(r"[\s\-]+", "_", str(l).strip().lower()).strip("_")
    if l.startswith("not_"): return "not_in_language"
    if l in ("mixed_feelings", "mixed", "mixed_feeling"): return "mixed_feelings"
    if l in ("unknown_state", "unknown"): return "unknown_state"
    return l

def norm_text(t):
    return re.sub(r"\s+", " ", str(t).strip().lower())

def read_local(path):
    with open(path, encoding="utf-8", errors="replace") as fh:
        raw_lines = [ln.strip() for ln in fh if ln.strip()]
    best_rows = []
    for sep in ("\t", ";", ","):
        rows = []
        for ln in raw_lines:
            parts = [p.strip() for p in ln.split(sep)]
            while parts and parts[-1] == "": parts.pop()
            if len(parts) < 2: continue
            lbl = norm_label(parts[-1])
            if lbl not in CLASSES: continue
            txt_parts = parts[1:-1] if (len(parts) >= 3 and re.fullmatch(r"\d+", parts[0])) else parts[:-1]
            txt = sep.join(txt_parts).strip()
            if txt: rows.append((txt, lbl))
        if len(rows) > len(best_rows): best_rows = rows
    return pd.DataFrame(best_rows, columns=["text", "label"])

def fetch_dataset_if_missing():
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    if any(p.suffix.lower() in (".tsv", ".csv", ".txt") for p in DATA_DIR.rglob("*")):
        return
    import urllib.request
    zpath = DATA_DIR / "dcm.zip"
    url = "https://zenodo.org/records/4750858/files/DravidianCodeMix-2020.zip?download=1"
    print("Downloading Zenodo dataset:", url)
    urllib.request.urlretrieve(url, zpath)
    with zipfile.ZipFile(zpath) as z: z.extractall(DATA_DIR)

def find_files(lang):
    fetch_dataset_if_missing()
    found = {}
    for p in sorted(DATA_DIR.rglob("*")):
        n = p.name.lower()
        if "__macosx" in [x.lower() for x in p.parts] or n.startswith("."): continue
        if p.suffix.lower() not in (".tsv", ".csv", ".txt") or lang not in n or "offens" in n or "without_label" in n: continue
        for split, keys in [("train", ["train"]), ("dev", ["dev", "val"]), ("test", ["test"])]:
            if any(k in n for k in keys):
                if split not in found or ("sentiment" in n and "sentiment" not in found[split].name.lower()):
                    found[split] = p
    return found

DATA = {}
for lang in LANGS:
    files = find_files(lang)
    splits = {s: read_local(p) for s, p in files.items()}
    tr, dv, te = splits["train"], splits["dev"], splits["test"]
    tr = tr.assign(_k=tr.text.map(norm_text)).drop_duplicates("_k")
    held = set(dv.text.map(norm_text)) | set(te.text.map(norm_text))
    tr = tr[~tr._k.isin(held)].drop(columns="_k").reset_index(drop=True)
    for d in (tr, dv, te): d["y"] = d.label.map(C2I)
    DATA[lang] = {"train": tr, "dev": dv, "test": te}
    print(f"[{lang}] train={len(tr)} | dev={len(dv)} | test={len(te)}")

VOWELS = set("aeiou")
TRANSLIT_RULES = [
    ("aa", "a"), ("a", "aa"), ("ee", "i"), ("oo", "u"), ("u", "oo"),
    ("th", "t"), ("zh", "l"), ("sh", "s"), ("w", "v"), ("v", "w"),
    ("ai", "ei"), ("ph", "f"),
]
def _is_ascii_word(w): return w.isascii() and any(ch.isalpha() for ch in w)
def n_elongate(t, rng, p=0.15):
    out = []
    for ch in t:
        out.append(ch)
        if ch.lower() in VOWELS and ch.isascii() and rng.random() < p:
            out.append(ch * int(rng.integers(1, 3)))
    return "".join(out)
def n_typo(t, rng, p=0.05):
    chars, i, out = list(t), 0, []
    while i < len(chars):
        ch = chars[i]
        if ch.isascii() and ch.isalpha() and rng.random() < p:
            op = rng.integers(0, 3)
            if op == 0: pass
            elif op == 1 and i + 1 < len(chars): out.extend([chars[i + 1], ch]); i += 1
            else: out.extend([ch, ch])
        else: out.append(ch)
        i += 1
    return "".join(out)
def n_translit(t, rng, p_word=0.5):
    words = t.split(" ")
    for j, w in enumerate(words):
        if _is_ascii_word(w) and rng.random() < p_word:
            lw = w.lower()
            cand = [(a, b) for a, b in TRANSLIT_RULES if a in lw]
            if cand:
                a, b = cand[int(rng.integers(0, len(cand)))]
                words[j] = lw.replace(a, b)
    return " ".join(words)
def n_combined(t, rng): return n_typo(n_elongate(n_translit(t, rng), rng), rng)

NOISES = {"noise_elong": n_elongate, "noise_typo": n_typo, "noise_translit": n_translit, "noise_combined": n_combined}
EVAL_SETS = {}
for lang in LANGS:
    te = DATA[lang]["test"]; texts, ys = te.text.tolist(), te.y.values
    sets = {"clean": (texts, ys)}
    for name, fn in NOISES.items():
        rng = np.random.default_rng(NOISE_SEED)
        sets[name] = ([fn(t, rng) for t in texts], ys)
    for other in LANGS:
        if other != lang:
            o = DATA[other]["test"]; sets[f"xl_{other}"] = (o.text.tolist(), o.y.values)
    EVAL_SETS[lang] = sets

# ---------------- 4. Patched Model, Training & Resumable Orchestrator ----------------
def fast_macro_f1(y, p, k=K):
    cm = np.bincount(y * k + p, minlength=k * k).reshape(k, k)
    tp = np.diag(cm).astype(float); fp = cm.sum(0) - tp; fn = cm.sum(1) - tp
    denom = 2 * tp + fp + fn
    return float(np.mean(np.where(denom > 0, 2 * tp / np.maximum(denom, 1), 0.0)))

def compute_metrics(y, p):
    return {
        "macro_f1":     float(f1_score(y, p, average="macro",    labels=list(range(K)), zero_division=0)),
        "weighted_f1":  float(f1_score(y, p, average="weighted", labels=list(range(K)), zero_division=0)),
        "acc":          float(accuracy_score(y, p)),
        "per_class_f1": f1_score(y, p, average=None, labels=list(range(K)), zero_division=0).tolist(),
    }

def class_weights(y):
    counts = np.bincount(y, minlength=K)
    return len(y) / (K * np.maximum(counts, 1))

def ros_indices(y, frac, rng):
    counts = np.bincount(y, minlength=K); target = int(frac * counts.max())
    idx = list(range(len(y)))
    for c in range(K):
        if 0 < counts[c] < target:
            idx.extend(rng.choice(np.where(y == c)[0], target - counts[c], replace=True).tolist())
    return np.array(idx)

def focal_loss(logits, y, gamma=FOCAL_GAMMA, weight=None):
    logp = F.log_softmax(logits.float(), -1)
    logpt = logp.gather(1, y[:, None]).squeeze(1); pt = logpt.exp()
    loss = -((1 - pt) ** gamma) * logpt
    if weight is not None: loss = loss * weight[y]
    return loss.mean()

_TOK, _FROZEN, _ENC_PARAMS = {}, {}, {}
def get_tok(model_key):
    if model_key not in _TOK: _TOK[model_key] = AutoTokenizer.from_pretrained(MODELS[model_key])
    return _TOK[model_key]

def lr_for(tuning, lr_override=None):
    return lr_override if lr_override is not None else (LR["full"] if tuning == "full" else LR["lora"])

def build_model(model_key, tuning):
    m = AutoModelForSequenceClassification.from_pretrained(
        MODELS[model_key], num_labels=K, id2label=dict(enumerate(CLASSES)), label2id=C2I
    )
    if tuning.startswith("lora"):
        from peft import LoraConfig, get_peft_model, TaskType
        p = TUNINGS[tuning]
        save_mods = ["classifier", "pooler"] if hasattr(m, "bert") else ["classifier"]
        cfg = LoraConfig(
            task_type=TaskType.SEQ_CLS, r=p["r"], lora_alpha=p["alpha"], lora_dropout=0.1,
            target_modules=["query", "value"], modules_to_save=save_mods,
        )
        m = get_peft_model(m, cfg)
    return m.to(DEVICE)

@torch.no_grad()
def predict(model, tok, texts, bs=128):
    model.eval()
    order = np.argsort([len(t) for t in texts]); out = np.zeros(len(texts), dtype=int)
    for i in range(0, len(texts), bs):
        ids = order[i:i + bs]
        enc = tok([texts[j] for j in ids], truncation=True, max_length=MAX_LEN, padding=True, return_tensors="pt").to(DEVICE)
        with torch.autocast("cuda", dtype=torch.float16):
            logits = model(**enc).logits
        out[ids] = logits.argmax(-1).cpu().numpy()
    return out

def train_finetune(lang, model_key, tuning, imb, seed, ros_frac=ROS_FRAC, lr_override=None):
    _FROZEN.clear(); gc.collect(); torch.cuda.empty_cache()
    set_seed(seed); rng = np.random.default_rng(seed)
    tr, dv = DATA[lang]["train"], DATA[lang]["dev"]
    texts, y = tr.text.tolist(), tr.y.values
    base_imb = imb.split("_f")[0]
    idx0 = ros_indices(y, ros_frac, rng) if base_imb == "ros" else np.arange(len(y))
    weight = torch.tensor(class_weights(y), dtype=torch.float, device=DEVICE) if base_imb == "wce" else None

    tok = get_tok(model_key); model = build_model(model_key, tuning)
    n_train = sum(p.numel() for p in model.parameters() if p.requires_grad)
    n_total = sum(p.numel() for p in model.parameters())
    used_lr = lr_for(tuning, lr_override)
    opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=used_lr, weight_decay=0.01)
    steps = EPOCHS * int(np.ceil(len(idx0) / BS))
    sched = get_linear_schedule_with_warmup(opt, int(0.06 * steps), steps)
    scaler = torch.amp.GradScaler("cuda")
    torch.cuda.reset_peak_memory_stats()

    train_time, best_f1, best_ep, best_state = 0.0, -1.0, -1, None
    for ep in range(EPOCHS):
        model.train(); perm = rng.permutation(idx0)
        torch.cuda.synchronize(); t0 = time.time()
        for i in range(0, len(perm), BS):
            b = perm[i:i + BS]
            enc = tok([texts[j] for j in b], truncation=True, max_length=MAX_LEN, padding=True, return_tensors="pt").to(DEVICE)
            yb = torch.tensor(y[b], dtype=torch.long, device=DEVICE)
            with torch.autocast("cuda", dtype=torch.float16):
                logits = model(**enc).logits
            if base_imb == "focal": loss = focal_loss(logits, yb)
            elif base_imb == "wce": loss = F.cross_entropy(logits.float(), yb, weight=weight)
            else: loss = F.cross_entropy(logits.float(), yb)
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt); scaler.update(); sched.step()
        torch.cuda.synchronize(); train_time += time.time() - t0
        dev_f1 = fast_macro_f1(dv.y.values, predict(model, tok, dv.text.tolist()))
        print(f"    ep{ep + 1} dev macro-F1 = {dev_f1:.4f}")
        if dev_f1 > best_f1:
            best_f1, best_ep = dev_f1, ep + 1
            best_state = {n: p.detach().cpu().clone() for n, p in model.named_parameters() if p.requires_grad}

    with torch.no_grad():
        for n, p in model.named_parameters():
            if n in best_state: p.copy_(best_state[n].to(p.device))
    res_sets, saved_preds = {}, {}
    for name, (tx, ys) in EVAL_SETS[lang].items():
        pr = predict(model, tok, tx); res_sets[name] = compute_metrics(ys, pr)
        if name in ("clean", "noise_combined"): saved_preds[name] = pr

    peak = torch.cuda.max_memory_allocated() / 1e9
    info = dict(
        trainable_params=n_train, total_params=n_total,
        trainable_pct=100 * n_train / n_total, train_time_s=train_time,
        peak_mem_gb=peak, best_epoch=best_ep, dev_macro_f1=best_f1,
        lr_used=used_lr, ros_frac=ros_frac
    )
    del model, opt, sched, best_state; gc.collect(); torch.cuda.empty_cache()
    return info, res_sets, saved_preds

@torch.no_grad()
def embed(model_key, texts, key, bs=128):
    f = WORK / "emb" / f"{model_key}_{key}.npy"
    if f.exists(): return np.load(f)
    if model_key not in _FROZEN:
        enc_m = AutoModel.from_pretrained(MODELS[model_key]).to(DEVICE).eval().half()
        _FROZEN[model_key] = enc_m
        _ENC_PARAMS[model_key] = sum(p.numel() for p in enc_m.parameters())
    enc_model, tok = _FROZEN[model_key], get_tok(model_key)
    order = np.argsort([len(t) for t in texts])
    out = np.zeros((len(texts), enc_model.config.hidden_size), dtype=np.float32)
    for i in range(0, len(texts), bs):
        ids = order[i:i + bs]
        enc = tok([texts[j] for j in ids], truncation=True, max_length=MAX_LEN, padding=True, return_tensors="pt").to(DEVICE)
        h = enc_model(**enc).last_hidden_state.float()
        m = enc["attention_mask"].unsqueeze(-1).float()
        out[ids] = ((h * m).sum(1) / m.sum(1).clamp(min=1e-9)).cpu().numpy()
    np.save(f, out)
    return out

def get_enc_param_count(model_key):
    if model_key not in _ENC_PARAMS:
        if model_key in _FROZEN: _ENC_PARAMS[model_key] = sum(p.numel() for p in _FROZEN[model_key].parameters())
        else:
            tmp = AutoModel.from_pretrained(MODELS[model_key])
            _ENC_PARAMS[model_key] = sum(p.numel() for p in tmp.parameters())
            del tmp; gc.collect()
    return _ENC_PARAMS[model_key]

def train_frozen(lang, model_key, imb, seed, ros_frac=ROS_FRAC):
    set_seed(seed); rng = np.random.default_rng(seed)
    base = imb.replace("frozen_", "").split("_f")[0]
    tr, dv = DATA[lang]["train"], DATA[lang]["dev"]
    Xtr = embed(model_key, tr.text.tolist(), f"{lang}_train"); ytr = tr.y.values
    Xdv = embed(model_key, dv.text.tolist(), f"{lang}_dev")
    mu, sd = Xtr.mean(0), Xtr.std(0) + 1e-6
    z = lambda X: (X - mu) / sd

    if base == "ros":
        idx = ros_indices(ytr, ros_frac, rng); X, y = Xtr[idx], ytr[idx]
    elif base == "smote":
        from imblearn.over_sampling import SMOTE
        counts = np.bincount(ytr, minlength=K); target = int(ros_frac * counts.max())
        strat = {c: target for c in range(K) if 1 < counts[c] < target}
        if strat:
            k = max(1, min(5, int(min(counts[c] for c in strat)) - 1))
            X_z, y = SMOTE(sampling_strategy=strat, k_neighbors=k, random_state=seed).fit_resample(z(Xtr), ytr)
            X = X_z * sd + mu
        else: X, y = Xtr, ytr
    else: X, y = Xtr, ytr

    head = nn.Sequential(nn.Linear(Xtr.shape[1], 256), nn.ReLU(), nn.Dropout(0.2), nn.Linear(256, K)).to(DEVICE)
    n_head = sum(p.numel() for p in head.parameters())
    n_enc = get_enc_param_count(model_key)
    opt = torch.optim.AdamW(head.parameters(), lr=LR["head"], weight_decay=0.01)
    Xt = torch.tensor(z(X), dtype=torch.float32, device=DEVICE); yt = torch.tensor(y, dtype=torch.long, device=DEVICE)
    Xd = torch.tensor(z(Xdv), dtype=torch.float32, device=DEVICE)
    w = torch.tensor(class_weights(ytr), dtype=torch.float, device=DEVICE) if base == "wce" else None

    torch.cuda.reset_peak_memory_stats()
    best_f1, best_state, best_ep, t0 = -1.0, None, -1, time.time()
    for ep in range(20):
        head.train(); perm = torch.randperm(len(Xt), device=DEVICE)
        for i in range(0, len(perm), 256):
            b = perm[i:i + 256]
            loss = F.cross_entropy(head(Xt[b]), yt[b], weight=w)
            opt.zero_grad(); loss.backward(); opt.step()
        head.eval()
        with torch.no_grad(): f = fast_macro_f1(dv.y.values, head(Xd).argmax(-1).cpu().numpy())
        if f > best_f1: best_f1, best_ep, best_state = f, ep + 1, {k_: v.clone() for k_, v in head.state_dict().items()}
    train_time = time.time() - t0

    head.load_state_dict(best_state); head.eval()
    res_sets, saved_preds = {}, {}
    for name, (tx, ys) in EVAL_SETS[lang].items():
        Xe = embed(model_key, tx, f"{lang}_{name}")
        with torch.no_grad(): pr = head(torch.tensor(z(Xe), dtype=torch.float32, device=DEVICE)).argmax(-1).cpu().numpy()
        res_sets[name] = compute_metrics(ys, pr)
        if name in ("clean", "noise_combined"): saved_preds[name] = pr

    peak = torch.cuda.max_memory_allocated() / 1e9
    info = dict(
        trainable_params=n_head, total_params=n_head + n_enc,
        trainable_pct=100 * n_head / (n_head + n_enc), train_time_s=train_time,
        peak_mem_gb=peak, best_epoch=best_ep, dev_macro_f1=best_f1,
        lr_used=LR["head"], ros_frac=ros_frac
    )
    del head, opt, Xt, yt, Xd, best_state; gc.collect()
    return info, res_sets, saved_preds

def run_id(r): return f"{r['lang']}|{r['model']}|{r['tuning']}|{r['imb']}|{r['seed']}"
def done_ids():
    if not RESULTS.exists(): return set()
    return {json.loads(l)["run_id"] for l in open(RESULTS, encoding="utf-8") if l.strip()}

def execute(runs):
    done = done_ids()
    todo = [r for r in runs if run_id(r) not in done]
    print(f"{len(runs)} runs requested | {len(runs) - len(todo)} already done | {len(todo)} remaining")
    t_stage = time.time()
    for n, r in enumerate(todo, 1):
        rid = run_id(r); frac = r.get("ros_frac", ROS_FRAC)
        lr_val = LR["head"] if r["imb"].startswith("frozen") else lr_for(r["tuning"])
        print(f"[{n}/{len(todo)}] {rid} (lr={lr_val}, ros_frac={frac})")
        try:
            if r["imb"].startswith("frozen"):
                info, sets, preds = train_frozen(r["lang"], r["model"], r["imb"], r["seed"], ros_frac=frac)
            else:
                info, sets, preds = train_finetune(r["lang"], r["model"], r["tuning"], r["imb"], r["seed"], ros_frac=frac)
        except torch.cuda.OutOfMemoryError:
            print("  OOM encountered — skipping run."); gc.collect(); torch.cuda.empty_cache(); continue

        row = dict(run_id=rid, lang=r["lang"], model=r["model"], tuning=r["tuning"], imb=r["imb"], seed=r["seed"], **info)
        for name, m in sets.items():
            row[f"{name}__macro_f1"] = m["macro_f1"]; row[f"{name}__acc"] = m["acc"]; row[f"{name}__weighted_f1"] = m["weighted_f1"]
        for c, f in zip(CLASSES, sets["clean"]["per_class_f1"]): row[f"clean__f1_{c}"] = f

        np.save(WORK / "preds" / (rid.replace("|", "__") + ".npy"), preds["clean"])
        if "noise_combined" in preds:
            np.save(WORK / "preds" / (rid.replace("|", "__") + "__noise_combined.npy"), preds["noise_combined"])

        with open(RESULTS, "a", encoding="utf-8") as fh:
            fh.write(json.dumps(row) + "\n")
            fh.flush()
            os.fsync(fh.fileno())
        gc.collect(); torch.cuda.empty_cache()
        print(f"  -> clean macro-F1 = {row['clean__macro_f1']:.4f} | dev = {info['dev_macro_f1']:.4f} | "
              f"trainable {info['trainable_pct']:.2f}% | {info['train_time_s']:.0f}s | elapsed {(time.time()-t_stage)/60:.1f}m")

print(f"\nStorage: {STORAGE_ROOT} (Drive={USE_DRIVE}) | GPU: {torch.cuda.get_device_name(0)} | Locked LR: {LR}")
backup_now("cell2_restored")

Mounted at /content/drive
Unpacking /content/drive/MyDrive/dcm_bench_LATEST.zip into /content/drive/MyDrive...
Unpack complete.
[tamil] train=34787 | dev=4398 | test=4402
[kannada] train=5717 | dev=767 | test=768

Storage: /content/drive/MyDrive (Drive=True) | GPU: Tesla T4 | Locked LR: {'full': 5e-05, 'lora': 0.0005, 'head': 0.001}
[BACKUP SAVED] /content/drive/MyDrive/dcm_bench/backups/dcm_backup_cell2_restored.zip (0.33 MB) | Unique completed runs: 145


In [ ]:
backup_path = WORK / "results_pre_pooler_fix_backup.jsonl"
if not backup_path.exists() and RESULTS.exists():
    shutil.copy2(RESULTS, backup_path)
    kept, removed = [], 0
    for line in open(backup_path, encoding="utf-8"):
        if not line.strip():
            continue
        rec = json.loads(line)
        is_old_bert_lora = (
            rec["lang"] == "kannada"
            and rec["model"] in ("muril", "mbert")
            and rec["tuning"] in ("lora_r8", "lora_r32")
            and rec["imb"] == "ce"
        )
        if is_old_bert_lora:
            removed += 1
        else:
            kept.append(line)
    RESULTS.write_text("".join(kept), encoding="utf-8")
    print(f"Backed up old results -> {backup_path.name} | Purged {removed} old Kannada BERT LoRA runs.")

    plan_file = WORK / "PLAN.md"
    if plan_file.exists() and "Pooler Parity Fix" not in plan_file.read_text(encoding="utf-8"):
        with open(plan_file, "a", encoding="utf-8") as fh:
            fh.write(
                "\n## Methodological Note: BERT Pooler Parity Fix\n"
                "- Kannada dev diagnostic (seed=42) showed MuRIL LoRA collapsed to majority class (12.46% dev F1) "
                "because PEFT freezes bert.pooler by default, whereas XLM-R keeps its equivalent 768x768 dense layer trainable inside classifier.\n"
                "- Applied modules_to_save=['classifier', 'pooler'] uniformly to BERT-family models (mbert, muril).\n"
            )
else:
    print("Pooler fix already applied; resuming remaining runs safely.")

runs_k_fix = [
    dict(lang="kannada", model=m, tuning=t, imb="ce", seed=s)
    for m in ["muril", "mbert"]
    for t in ["lora_r8", "lora_r32"]
    for s in SEEDS
]
execute(runs_k_fix)

df_k = pd.read_json(RESULTS, lines=True).drop_duplicates("run_id", keep="last")
df_k = df_k[(df_k.lang == "kannada") & (df_k.imb == "ce")]
rows_k = []
for (m, t), g in df_k.groupby(["model", "tuning"]):
    rows_k.append(dict(
        model=m, tuning=t, n_seeds=len(g),
        dev_f1=f"{100*g.dev_macro_f1.mean():.2f} ± {100*g.dev_macro_f1.std(ddof=1):.2f}" if len(g) > 1 else f"{100*g.dev_macro_f1.mean():.2f}",
        clean_f1=f"{100*g.clean__macro_f1.mean():.2f} ± {100*g.clean__macro_f1.std(ddof=1):.2f}" if len(g) > 1 else f"{100*g.clean__macro_f1.mean():.2f}",
        xl_tamil=f"{100*g.xl_tamil__macro_f1.mean():.2f}" if "xl_tamil__macro_f1" in g else "n/a",
        comb_noise=f"{100*g.noise_combined__macro_f1.mean():.2f}",
        trainable_pct=round(g.trainable_pct.mean(), 3),
        time_s=round(g.train_time_s.mean(), 1),
        vram_gb=round(g.peak_mem_gb.mean(), 2),
    ))

print("\n=== UPDATED KANNADA STAGE-A SUMMARY ===")
print(pd.DataFrame(rows_k).sort_values(["model", "tuning"]).to_string(index=False))

backup_now("cell3_kannada_pooler_fix")

Backed up old results -> results_pre_pooler_fix_backup.jsonl | Purged 20 old Kannada BERT LoRA runs.
20 runs requested | 0 already done | 20 remaining
[1/20] kannada|muril|lora_r8|ce|13 (lr=0.0005, ros_frac=0.5)


config.json:   0%|          | 0.00/411 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/206 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/3.16M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/113 [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  953MB            

... (45 outputs left out)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

    ep1 dev macro-F1 = 0.3681
    ep2 dev macro-F1 = 0.4400
    ep3 dev macro-F1 = 0.4577
  -> clean macro-F1 = 0.4913 | dev = 0.4577 | trainable 0.99% | 40s | elapsed 15.4m
[20/20] kannada|mbert|lora_r32|ce|101 (lr=0.0005, ros_frac=0.5)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

    ep1 dev macro-F1 = 0.4156
    ep2 dev macro-F1 = 0.4503
    ep3 dev macro-F1 = 0.4598
  -> clean macro-F1 = 0.4888 | dev = 0.4598 | trainable 0.99% | 41s | elapsed 16.3m

=== UPDATED KANNADA STAGE-A SUMMARY ===
model   tuning  n_seeds        dev_f1      clean_f1 xl_tamil comb_noise  trainable_pct  time_s  vram_gb
mbert     full        5  47.43 ± 1.24  49.81 ± 1.24    28.65      43.48        100.000    82.7     3.96
mbert lora_r32        5  46.94 ± 1.21  47.88 ± 1.31    25.89      42.49          0.988    40.4     1.62
mbert  lora_r8        5  44.89 ± 1.91  46.96 ± 0.68    24.04      40.39          0.498    40.2     1.60
muril     full        5  34.78 ± 1.21  35.34 ± 1.15    21.24      31.76        100.000    95.1     4.91
muril lora_r32        5 28.29 ± 14.92 28.01 ± 14.23    20.44      25.87          0.741    38.1     1.86
muril  lora_r8        5  35.20 ± 1.02  36.66 ± 0.88    22.20      32.52          0.373    38.6     1.84
 xlmr     full        5 38.83 ± 11.78 39.49 ± 11.60    24

In [ ]:
# 1. Upgrade execute() to refresh My Drive/dcm_bench_LATEST.zip after EVERY completed run
def save_portable_zip():
    tmp_zip = Path("/content/_tmp_dcm_latest.zip")
    with zipfile.ZipFile(tmp_zip, "w", compression=zipfile.ZIP_DEFLATED) as zf:
        for fname in ["results.jsonl", "results_pre_pooler_fix_backup.jsonl", "lr_config.json", "PLAN.md", "best_config.json"]:
            fp = WORK / fname
            if fp.exists():
                zf.write(fp, arcname=f"dcm_bench/main/{fname}")
        for folder in ["preds", "tables", "figs"]:
            fdir = WORK / folder
            if fdir.exists():
                for fp in fdir.rglob("*"):
                    if fp.is_file():
                        zf.write(fp, arcname=f"dcm_bench/{fp.relative_to(BASE_DIR)}")
        if DATA_DIR.exists():
            for fp in DATA_DIR.rglob("*"):
                if fp.is_file() and fp.suffix.lower() in (".csv", ".tsv", ".txt"):
                    zf.write(fp, arcname=f"dcm_bench/{fp.relative_to(BASE_DIR)}")
    shutil.copy2(tmp_zip, STORAGE_ROOT / "dcm_bench_LATEST.zip")
    tmp_zip.unlink(missing_ok=True)

def execute(runs):
    done = done_ids()
    todo = [r for r in runs if run_id(r) not in done]
    print(f"{len(runs)} runs requested | {len(runs) - len(todo)} already done | {len(todo)} remaining")
    t_stage = time.time()
    for n, r in enumerate(todo, 1):
        rid = run_id(r)
        frac = r.get("ros_frac", ROS_FRAC)
        lr_val = LR["head"] if r["imb"].startswith("frozen") else lr_for(r["tuning"])
        print(f"[{n}/{len(todo)}] {rid} (lr={lr_val}, ros_frac={frac})")
        try:
            if r["imb"].startswith("frozen"):
                info, sets, preds = train_frozen(r["lang"], r["model"], r["imb"], r["seed"], ros_frac=frac)
            else:
                info, sets, preds = train_finetune(r["lang"], r["model"], r["tuning"], r["imb"], r["seed"], ros_frac=frac)
        except torch.cuda.OutOfMemoryError:
            print("  OOM encountered — skipping run."); gc.collect(); torch.cuda.empty_cache(); continue

        row = dict(run_id=rid, lang=r["lang"], model=r["model"], tuning=r["tuning"], imb=r["imb"], seed=r["seed"], **info)
        for name, m in sets.items():
            row[f"{name}__macro_f1"]    = m["macro_f1"]
            row[f"{name}__acc"]         = m["acc"]
            row[f"{name}__weighted_f1"] = m["weighted_f1"]
        for c, f in zip(CLASSES, sets["clean"]["per_class_f1"]):
            row[f"clean__f1_{c}"] = f

        np.save(WORK / "preds" / (rid.replace("|", "__") + ".npy"), preds["clean"])
        if "noise_combined" in preds:
            np.save(WORK / "preds" / (rid.replace("|", "__") + "__noise_combined.npy"), preds["noise_combined"])

        with open(RESULTS, "a", encoding="utf-8") as fh:
            fh.write(json.dumps(row) + "\n")
            fh.flush()
            os.fsync(fh.fileno())

        save_portable_zip()
        gc.collect(); torch.cuda.empty_cache()
        print(f"  -> clean macro-F1 = {row['clean__macro_f1']:.4f} | dev = {info['dev_macro_f1']:.4f} | "
              f"trainable {info['trainable_pct']:.2f}% | {info['train_time_s']:.0f}s | Drive zip updated | elapsed {(time.time()-t_stage)/60:.1f}m")

# Save current 60-run post-Cell-3 state to My Drive/dcm_bench_LATEST.zip immediately
save_portable_zip()
print("Portable Drive auto-save enabled -> My Drive/dcm_bench_LATEST.zip")

# 2. Run Stage A: Tamil mBERT (15 runs, ~85 min on T4)
runs_tamil_mbert = [
    dict(lang="tamil", model="mbert", tuning=t, imb="ce", seed=s)
    for t in TUNINGS
    for s in SEEDS
]
print(f"\nStarting Stage A (Tamil - mBERT): {len(runs_tamil_mbert)} runs")
execute(runs_tamil_mbert)

# 3. Print Tamil mBERT summary + milestone backup
df_tm = pd.read_json(RESULTS, lines=True).drop_duplicates("run_id", keep="last")
df_tm = df_tm[(df_tm.lang == "tamil") & (df_tm.model == "mbert") & (df_tm.imb == "ce")]
rows_tm = []
for t, g in df_tm.groupby("tuning"):
    rows_tm.append(dict(
        model="mbert", tuning=t, n_seeds=len(g),
        dev_f1=f"{100*g.dev_macro_f1.mean():.2f} ± {100*g.dev_macro_f1.std(ddof=1):.2f}" if len(g) > 1 else f"{100*g.dev_macro_f1.mean():.2f}",
        clean_f1=f"{100*g.clean__macro_f1.mean():.2f} ± {100*g.clean__macro_f1.std(ddof=1):.2f}" if len(g) > 1 else f"{100*g.clean__macro_f1.mean():.2f}",
        xl_kannada=f"{100*g.xl_kannada__macro_f1.mean():.2f}" if "xl_kannada__macro_f1" in g else "n/a",
        comb_noise=f"{100*g.noise_combined__macro_f1.mean():.2f}",
        trainable_pct=round(g.trainable_pct.mean(), 3),
        time_min=round(g.train_time_s.mean() / 60, 2),
        vram_gb=round(g.peak_mem_gb.mean(), 2),
    ))

print("\n=== TAMIL STAGE-A SUMMARY (mBERT) ===")
print(pd.DataFrame(rows_tm).sort_values("tuning").to_string(index=False))

backup_now("cell4_tamil_mbert")

Portable Drive auto-save enabled -> My Drive/dcm_bench_LATEST.zip

Starting Stage A (Tamil - mBERT): 15 runs
15 runs requested | 0 already done | 15 remaining
[1/15] tamil|mbert|full|ce|13 (lr=5e-05, ros_frac=0.5)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

    ep1 dev macro-F1 = 0.3303
    ep2 dev macro-F1 = 0.4870
    ep3 dev macro-F1 = 0.4858
  -> clean macro-F1 = 0.4868 | dev = 0.4870 | trainable 100.00% | 514s | Drive zip updated | elapsed 9.0m
[2/15] tamil|mbert|full|ce|21 (lr=5e-05, ros_frac=0.5)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

    ep1 dev macro-F1 = 0.4688
    ep2 dev macro-F1 = 0.5017
    ep3 dev macro-F1 = 0.4983
  -> clean macro-F1 = 0.4950 | dev = 0.5017 | trainable 100.00% | 513s | Drive zip updated | elapsed 18.1m
[3/15] tamil|mbert|full|ce|42 (lr=5e-05, ros_frac=0.5)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

... (21 outputs left out)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

    ep1 dev macro-F1 = 0.4001
    ep2 dev macro-F1 = 0.4291
    ep3 dev macro-F1 = 0.4650
  -> clean macro-F1 = 0.4810 | dev = 0.4650 | trainable 0.99% | 248s | Drive zip updated | elapsed 86.6m
[15/15] tamil|mbert|lora_r32|ce|101 (lr=0.0005, ros_frac=0.5)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

    ep1 dev macro-F1 = 0.3977
    ep2 dev macro-F1 = 0.4391
    ep3 dev macro-F1 = 0.4491
  -> clean macro-F1 = 0.4643 | dev = 0.4491 | trainable 0.99% | 249s | Drive zip updated | elapsed 91.3m

=== TAMIL STAGE-A SUMMARY (mBERT) ===
model   tuning  n_seeds       dev_f1     clean_f1 xl_kannada comb_noise  trainable_pct  time_min  vram_gb
mbert     full        5 49.80 ± 0.66 49.15 ± 0.30      18.70      44.29        100.000      8.54     3.96
mbert lora_r32        5 46.10 ± 0.68 47.27 ± 0.88      17.62      42.99          0.988      4.17     1.62
mbert  lora_r8        5 44.49 ± 0.43 45.04 ± 0.42      18.03      41.07          0.498      4.12     1.60
[BACKUP SAVED] /content/drive/MyDrive/dcm_bench/backups/dcm_backup_cell4_tamil_mbert.zip (0.21 MB) | Unique completed runs: 75


In [ ]:
# 1. Upgrade execute() to refresh My Drive/dcm_bench_LATEST.zip after EVERY completed run
def save_portable_zip():
    tmp_zip = Path("/content/_tmp_dcm_latest.zip")
    with zipfile.ZipFile(tmp_zip, "w", compression=zipfile.ZIP_DEFLATED) as zf:
        for fname in ["results.jsonl", "results_pre_pooler_fix_backup.jsonl", "lr_config.json", "PLAN.md", "best_config.json"]:
            fp = WORK / fname
            if fp.exists():
                zf.write(fp, arcname=f"dcm_bench/main/{fname}")
        for folder in ["preds", "tables", "figs"]:
            fdir = WORK / folder
            if fdir.exists():
                for fp in fdir.rglob("*"):
                    if fp.is_file():
                        zf.write(fp, arcname=f"dcm_bench/{fp.relative_to(BASE_DIR)}")
        if DATA_DIR.exists():
            for fp in DATA_DIR.rglob("*"):
                if fp.is_file() and fp.suffix.lower() in (".csv", ".tsv", ".txt"):
                    zf.write(fp, arcname=f"dcm_bench/{fp.relative_to(BASE_DIR)}")
    shutil.copy2(tmp_zip, STORAGE_ROOT / "dcm_bench_LATEST.zip")
    tmp_zip.unlink(missing_ok=True)

def execute(runs):
    done = done_ids()
    todo = [r for r in runs if run_id(r) not in done]
    print(f"{len(runs)} runs requested | {len(runs) - len(todo)} already done | {len(todo)} remaining")
    t_stage = time.time()
    for n, r in enumerate(todo, 1):
        rid = run_id(r)
        frac = r.get("ros_frac", ROS_FRAC)
        lr_val = LR["head"] if r["imb"].startswith("frozen") else lr_for(r["tuning"])
        print(f"[{n}/{len(todo)}] {rid} (lr={lr_val}, ros_frac={frac})")
        try:
            if r["imb"].startswith("frozen"):
                info, sets, preds = train_frozen(r["lang"], r["model"], r["imb"], r["seed"], ros_frac=frac)
            else:
                info, sets, preds = train_finetune(r["lang"], r["model"], r["tuning"], r["imb"], r["seed"], ros_frac=frac)
        except torch.cuda.OutOfMemoryError:
            print("  OOM encountered — skipping run."); gc.collect(); torch.cuda.empty_cache(); continue

        row = dict(run_id=rid, lang=r["lang"], model=r["model"], tuning=r["tuning"], imb=r["imb"], seed=r["seed"], **info)
        for name, m in sets.items():
            row[f"{name}__macro_f1"]    = m["macro_f1"]
            row[f"{name}__acc"]         = m["acc"]
            row[f"{name}__weighted_f1"] = m["weighted_f1"]
        for c, f in zip(CLASSES, sets["clean"]["per_class_f1"]):
            row[f"clean__f1_{c}"] = f

        np.save(WORK / "preds" / (rid.replace("|", "__") + ".npy"), preds["clean"])
        if "noise_combined" in preds:
            np.save(WORK / "preds" / (rid.replace("|", "__") + "__noise_combined.npy"), preds["noise_combined"])

        with open(RESULTS, "a", encoding="utf-8") as fh:
            fh.write(json.dumps(row) + "\n")
            fh.flush()
            os.fsync(fh.fileno())

        save_portable_zip()
        gc.collect(); torch.cuda.empty_cache()
        print(f"  -> clean macro-F1 = {row['clean__macro_f1']:.4f} | dev = {info['dev_macro_f1']:.4f} | "
              f"trainable {info['trainable_pct']:.2f}% | {info['train_time_s']:.0f}s | Drive zip updated | elapsed {(time.time()-t_stage)/60:.1f}m")

# Save current 60-run post-Cell-3 state to My Drive/dcm_bench_LATEST.zip immediately
save_portable_zip()
print("Portable Drive auto-save enabled -> My Drive/dcm_bench_LATEST.zip")

# 2. Run Stage A: Tamil mBERT (15 runs, ~85 min on T4)
runs_tamil_mbert = [
    dict(lang="tamil", model="mbert", tuning=t, imb="ce", seed=s)
    for t in TUNINGS
    for s in SEEDS
]
print(f"\nStarting Stage A (Tamil - mBERT): {len(runs_tamil_mbert)} runs")
execute(runs_tamil_mbert)

# 3. Print Tamil mBERT summary + milestone backup
df_tm = pd.read_json(RESULTS, lines=True).drop_duplicates("run_id", keep="last")
df_tm = df_tm[(df_tm.lang == "tamil") & (df_tm.model == "mbert") & (df_tm.imb == "ce")]
rows_tm = []
for t, g in df_tm.groupby("tuning"):
    rows_tm.append(dict(
        model="mbert", tuning=t, n_seeds=len(g),
        dev_f1=f"{100*g.dev_macro_f1.mean():.2f} ± {100*g.dev_macro_f1.std(ddof=1):.2f}" if len(g) > 1 else f"{100*g.dev_macro_f1.mean():.2f}",
        clean_f1=f"{100*g.clean__macro_f1.mean():.2f} ± {100*g.clean__macro_f1.std(ddof=1):.2f}" if len(g) > 1 else f"{100*g.clean__macro_f1.mean():.2f}",
        xl_kannada=f"{100*g.xl_kannada__macro_f1.mean():.2f}" if "xl_kannada__macro_f1" in g else "n/a",
        comb_noise=f"{100*g.noise_combined__macro_f1.mean():.2f}",
        trainable_pct=round(g.trainable_pct.mean(), 3),
        time_min=round(g.train_time_s.mean() / 60, 2),
        vram_gb=round(g.peak_mem_gb.mean(), 2),
    ))

print("\n=== TAMIL STAGE-A SUMMARY (mBERT) ===")
print(pd.DataFrame(rows_tm).sort_values("tuning").to_string(index=False))

backup_now("cell4_tamil_mbert")

Portable Drive auto-save enabled -> My Drive/dcm_bench_LATEST.zip

Starting Stage A (Tamil - mBERT): 15 runs
15 runs requested | 15 already done | 0 remaining

=== TAMIL STAGE-A SUMMARY (mBERT) ===
model   tuning  n_seeds       dev_f1     clean_f1 xl_kannada comb_noise  trainable_pct  time_min  vram_gb
mbert     full        5 49.80 ± 0.66 49.15 ± 0.30      18.70      44.29        100.000      8.54     3.96
mbert lora_r32        5 46.10 ± 0.68 47.27 ± 0.88      17.62      42.99          0.988      4.17     1.62
mbert  lora_r8        5 44.49 ± 0.43 45.04 ± 0.42      18.03      41.07          0.498      4.12     1.60
[BACKUP SAVED] /content/drive/MyDrive/dcm_bench/backups/dcm_backup_cell4_tamil_mbert.zip (0.21 MB) | Unique completed runs: 75


In [ ]:
# Ensure per-run Drive zip auto-save is active even after a restart or account switch
def save_portable_zip():
    tmp_zip = Path("/content/_tmp_dcm_latest.zip")
    with zipfile.ZipFile(tmp_zip, "w", compression=zipfile.ZIP_DEFLATED) as zf:
        for fname in ["results.jsonl", "results_pre_pooler_fix_backup.jsonl", "lr_config.json", "PLAN.md", "best_config.json"]:
            fp = WORK / fname
            if fp.exists():
                zf.write(fp, arcname=f"dcm_bench/main/{fname}")
        for folder in ["preds", "tables", "figs"]:
            fdir = WORK / folder
            if fdir.exists():
                for fp in fdir.rglob("*"):
                    if fp.is_file():
                        zf.write(fp, arcname=f"dcm_bench/{fp.relative_to(BASE_DIR)}")
        if DATA_DIR.exists():
            for fp in DATA_DIR.rglob("*"):
                if fp.is_file() and fp.suffix.lower() in (".csv", ".tsv", ".txt"):
                    zf.write(fp, arcname=f"dcm_bench/{fp.relative_to(BASE_DIR)}")
    shutil.copy2(tmp_zip, STORAGE_ROOT / "dcm_bench_LATEST.zip")
    tmp_zip.unlink(missing_ok=True)

def execute(runs):
    done = done_ids()
    todo = [r for r in runs if run_id(r) not in done]
    print(f"{len(runs)} runs requested | {len(runs) - len(todo)} already done | {len(todo)} remaining")
    t_stage = time.time()
    for n, r in enumerate(todo, 1):
        rid = run_id(r)
        frac = r.get("ros_frac", ROS_FRAC)
        lr_val = LR["head"] if r["imb"].startswith("frozen") else lr_for(r["tuning"])
        print(f"[{n}/{len(todo)}] {rid} (lr={lr_val}, ros_frac={frac})")
        try:
            if r["imb"].startswith("frozen"):
                info, sets, preds = train_frozen(r["lang"], r["model"], r["imb"], r["seed"], ros_frac=frac)
            else:
                info, sets, preds = train_finetune(r["lang"], r["model"], r["tuning"], r["imb"], r["seed"], ros_frac=frac)
        except torch.cuda.OutOfMemoryError:
            print("  OOM encountered — skipping run."); gc.collect(); torch.cuda.empty_cache(); continue

        row = dict(run_id=rid, lang=r["lang"], model=r["model"], tuning=r["tuning"], imb=r["imb"], seed=r["seed"], **info)
        for name, m in sets.items():
            row[f"{name}__macro_f1"]    = m["macro_f1"]
            row[f"{name}__acc"]         = m["acc"]
            row[f"{name}__weighted_f1"] = m["weighted_f1"]
        for c, f in zip(CLASSES, sets["clean"]["per_class_f1"]):
            row[f"clean__f1_{c}"] = f

        np.save(WORK / "preds" / (rid.replace("|", "__") + ".npy"), preds["clean"])
        if "noise_combined" in preds:
            np.save(WORK / "preds" / (rid.replace("|", "__") + "__noise_combined.npy"), preds["noise_combined"])

        with open(RESULTS, "a", encoding="utf-8") as fh:
            fh.write(json.dumps(row) + "\n")
            fh.flush()
            os.fsync(fh.fileno())

        save_portable_zip()
        gc.collect(); torch.cuda.empty_cache()
        print(f"  -> clean macro-F1 = {row['clean__macro_f1']:.4f} | dev = {info['dev_macro_f1']:.4f} | "
              f"trainable {info['trainable_pct']:.2f}% | {info['train_time_s']:.0f}s | Drive zip updated | elapsed {(time.time()-t_stage)/60:.1f}m")

n_pre = len(done_ids())
print(f"Verified unique runs before Cell 5: {n_pre} (expected >= 75)")
assert n_pre >= 75, f"Expected at least 75 completed runs, found {n_pre}. Check {RESULTS}."

# 1. Run final 15 Stage-A experiments: Tamil MuRIL
runs_tamil_muril = [
    dict(lang="tamil", model="muril", tuning=t, imb="ce", seed=s)
    for t in TUNINGS
    for s in SEEDS
]
print(f"\nStarting Stage A (Tamil - MuRIL): {len(runs_tamil_muril)} runs")
execute(runs_tamil_muril)

# 2. Tamil MuRIL summary table
results_df = pd.read_json(RESULTS, lines=True).drop_duplicates("run_id", keep="last")
df_tmur = results_df[
    (results_df.lang == "tamil") & (results_df.model == "muril") & (results_df.imb == "ce")
]
rows_tmur = []
for t, g in df_tmur.groupby("tuning"):
    rows_tmur.append(dict(
        model="muril", tuning=t, n_seeds=len(g),
        dev_f1=f"{100*g.dev_macro_f1.mean():.2f} ± {100*g.dev_macro_f1.std(ddof=1):.2f}" if len(g) > 1 else f"{100*g.dev_macro_f1.mean():.2f}",
        clean_f1=f"{100*g.clean__macro_f1.mean():.2f} ± {100*g.clean__macro_f1.std(ddof=1):.2f}" if len(g) > 1 else f"{100*g.clean__macro_f1.mean():.2f}",
        xl_kannada=f"{100*g.xl_kannada__macro_f1.mean():.2f}" if "xl_kannada__macro_f1" in g else "n/a",
        comb_noise=f"{100*g.noise_combined__macro_f1.mean():.2f}",
        trainable_pct=round(g.trainable_pct.mean(), 3),
        time_min=round(g.train_time_s.mean() / 60, 2),
        vram_gb=round(g.peak_mem_gb.mean(), 2),
    ))
print("\n=== TAMIL STAGE-A SUMMARY (MuRIL) ===")
print(pd.DataFrame(rows_tmur).sort_values("tuning").to_string(index=False))

# 3. Verify 90/90 Stage-A runs & select BEST strictly on DEV split
stage_a_df = results_df[results_df.imb == "ce"].copy()
print(f"\nTotal Stage-A runs completed: {len(stage_a_df)} / 90")

per_lang_dev = (
    stage_a_df.groupby(["model", "tuning", "lang"])["dev_macro_f1"]
    .mean().unstack("lang").mul(100).round(2)
)
dev_table = (
    stage_a_df.groupby(["model", "tuning"])
    .agg(
        n_runs=("dev_macro_f1", "count"),
        mean_dev_f1=("dev_macro_f1", "mean"),
        trainable_pct=("trainable_pct", "mean"),
        mean_time_min=("train_time_s", lambda s: s.mean() / 60),
        peak_vram_gb=("peak_mem_gb", "mean"),
    )
    .join(per_lang_dev)
    .reset_index()
    .sort_values("mean_dev_f1", ascending=False)
    .reset_index(drop=True)
)
dev_table["mean_dev_f1_pct"] = (100 * dev_table["mean_dev_f1"]).round(2)
dev_table["trainable_pct"]   = dev_table["trainable_pct"].round(3)
dev_table["mean_time_min"]   = dev_table["mean_time_min"].round(2)
dev_table["peak_vram_gb"]    = dev_table["peak_vram_gb"].round(2)

display_cols = [
    "model", "tuning", "n_runs", "tamil", "kannada",
    "mean_dev_f1_pct", "trainable_pct", "mean_time_min", "peak_vram_gb",
]
print("\n=== STAGE-A DEV-ONLY SELECTION TABLE (TEST SPLIT UNTOUCHED) ===")
print(dev_table[display_cols].to_string(index=False))

top_dev = dev_table["mean_dev_f1"].max()
eligible = dev_table[dev_table["mean_dev_f1"] >= (top_dev - 0.01)].sort_values(
    ["trainable_pct", "mean_time_min"], ascending=[True, True]
)
chosen = eligible.iloc[0]
BEST = dict(model=str(chosen["model"]), tuning=str(chosen["tuning"]))
decision_note = (
    f"Selected on DEV only: {BEST} "
    f"(mean dev macro-F1 = {chosen['mean_dev_f1_pct']:.2f}%, "
    f"within 1.0 pt of top {100 * top_dev:.2f}%, "
    f"trainable_pct = {chosen['trainable_pct']:.3f}%, "
    f"mean time = {chosen['mean_time_min']:.2f} min)."
)
print("\n>>>", decision_note)

(WORK / "best_config.json").write_text(
    json.dumps({"BEST": BEST, "reason": decision_note}, indent=2),
    encoding="utf-8",
)
dev_table[display_cols].to_csv(WORK / "tables" / "T0_dev_selection.csv", index=False)

plan_file = WORK / "PLAN.md"
if plan_file.exists():
    plan_txt = plan_file.read_text(encoding="utf-8")
    if "## BEST Configuration Decision" not in plan_txt:
        plan_file.write_text(plan_txt + f"\n## BEST Configuration Decision\n- {decision_note}\n", encoding="utf-8")

save_portable_zip()
backup_now("cell5_stageA_complete")

Verified unique runs before Cell 5: 75 (expected >= 75)

Starting Stage A (Tamil - MuRIL): 15 runs
15 runs requested | 0 already done | 15 remaining
[1/15] tamil|muril|full|ce|13 (lr=5e-05, ros_frac=0.5)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

    ep1 dev macro-F1 = 0.1430
    ep2 dev macro-F1 = 0.1430
    ep3 dev macro-F1 = 0.1430
  -> clean macro-F1 = 0.1450 | dev = 0.1430 | trainable 100.00% | 582s | Drive zip updated | elapsed 10.1m
[2/15] tamil|muril|full|ce|21 (lr=5e-05, ros_frac=0.5)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

    ep1 dev macro-F1 = 0.1430
    ep2 dev macro-F1 = 0.1430
    ep3 dev macro-F1 = 0.1430
  -> clean macro-F1 = 0.1450 | dev = 0.1430 | trainable 100.00% | 582s | Drive zip updated | elapsed 20.3m
[3/15] tamil|muril|full|ce|42 (lr=5e-05, ros_frac=0.5)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

... (21 outputs left out)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

    ep1 dev macro-F1 = 0.2589
    ep2 dev macro-F1 = 0.4141
    ep3 dev macro-F1 = 0.4531
  -> clean macro-F1 = 0.4511 | dev = 0.4531 | trainable 0.74% | 235s | Drive zip updated | elapsed 89.8m
[15/15] tamil|muril|lora_r32|ce|101 (lr=0.0005, ros_frac=0.5)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

    ep1 dev macro-F1 = 0.1430
    ep2 dev macro-F1 = 0.1430
    ep3 dev macro-F1 = 0.1430
  -> clean macro-F1 = 0.1450 | dev = 0.1430 | trainable 0.74% | 234s | Drive zip updated | elapsed 94.1m

=== TAMIL STAGE-A SUMMARY (MuRIL) ===
model   tuning  n_seeds        dev_f1      clean_f1 xl_kannada comb_noise  trainable_pct  time_min  vram_gb
muril     full        5  14.30 ± 0.00  14.50 ± 0.00      12.72      14.50        100.000      9.70     4.91
muril lora_r32        5 20.51 ± 13.87 20.62 ± 13.69      14.94      20.11          0.741      3.91     1.86
muril  lora_r8        5 36.79 ± 12.93 37.92 ± 13.41      19.22      35.00          0.373      3.88     1.84

Total Stage-A runs completed: 90 / 90

=== STAGE-A DEV-ONLY SELECTION TABLE (TEST SPLIT UNTOUCHED) ===
model   tuning  n_runs  tamil  kannada  mean_dev_f1_pct  trainable_pct  mean_time_min  peak_vram_gb
mbert     full      10  49.80    47.43            48.61        100.000           4.96          3.96
mbert lora_r32      10  46.10 

In [ ]:
# 1. Ensure output folders & per-run Drive zip auto-save are active (safe after restart/account switch)
for sub in ["preds", "emb", "tables", "figs"]:
    (WORK / sub).mkdir(parents=True, exist_ok=True)

def save_portable_zip():
    tmp_zip = Path("/content/_tmp_dcm_latest.zip")
    with zipfile.ZipFile(tmp_zip, "w", compression=zipfile.ZIP_DEFLATED) as zf:
        for fname in ["results.jsonl", "results_pre_pooler_fix_backup.jsonl", "lr_config.json", "PLAN.md", "best_config.json"]:
            fp = WORK / fname
            if fp.exists():
                zf.write(fp, arcname=f"dcm_bench/main/{fname}")
        for folder in ["preds", "tables", "figs"]:
            fdir = WORK / folder
            if fdir.exists():
                for fp in fdir.rglob("*"):
                    if fp.is_file():
                        zf.write(fp, arcname=f"dcm_bench/{fp.relative_to(BASE_DIR)}")
        if DATA_DIR.exists():
            for fp in DATA_DIR.rglob("*"):
                if fp.is_file() and fp.suffix.lower() in (".csv", ".tsv", ".txt"):
                    zf.write(fp, arcname=f"dcm_bench/{fp.relative_to(BASE_DIR)}")
    shutil.copy2(tmp_zip, STORAGE_ROOT / "dcm_bench_LATEST.zip")
    tmp_zip.unlink(missing_ok=True)

def execute(runs):
    done = done_ids()
    todo = [r for r in runs if run_id(r) not in done]
    print(f"{len(runs)} runs requested | {len(runs) - len(todo)} already done | {len(todo)} remaining")
    t_stage = time.time()
    for n, r in enumerate(todo, 1):
        rid = run_id(r)
        frac = r.get("ros_frac", ROS_FRAC)
        lr_val = LR["head"] if r["imb"].startswith("frozen") else lr_for(r["tuning"])
        print(f"[{n}/{len(todo)}] {rid} (lr={lr_val}, ros_frac={frac})")
        try:
            if r["imb"].startswith("frozen"):
                info, sets, preds = train_frozen(r["lang"], r["model"], r["imb"], r["seed"], ros_frac=frac)
            else:
                info, sets, preds = train_finetune(r["lang"], r["model"], r["tuning"], r["imb"], r["seed"], ros_frac=frac)
        except torch.cuda.OutOfMemoryError:
            print("  OOM encountered — skipping run."); gc.collect(); torch.cuda.empty_cache(); continue

        row = dict(run_id=rid, lang=r["lang"], model=r["model"], tuning=r["tuning"], imb=r["imb"], seed=r["seed"], **info)
        for name, m in sets.items():
            row[f"{name}__macro_f1"]    = m["macro_f1"]
            row[f"{name}__acc"]         = m["acc"]
            row[f"{name}__weighted_f1"] = m["weighted_f1"]
        for c, f in zip(CLASSES, sets["clean"]["per_class_f1"]):
            row[f"clean__f1_{c}"] = f

        np.save(WORK / "preds" / (rid.replace("|", "__") + ".npy"), preds["clean"])
        if "noise_combined" in preds:
            np.save(WORK / "preds" / (rid.replace("|", "__") + "__noise_combined.npy"), preds["noise_combined"])

        with open(RESULTS, "a", encoding="utf-8") as fh:
            fh.write(json.dumps(row) + "\n")
            fh.flush()
            os.fsync(fh.fileno())

        save_portable_zip()
        gc.collect(); torch.cuda.empty_cache()
        print(f"  -> clean macro-F1 = {row['clean__macro_f1']:.4f} | dev = {info['dev_macro_f1']:.4f} | "
              f"trainable {info['trainable_pct']:.2f}% | {info['train_time_s']:.0f}s | Drive zip updated | elapsed {(time.time()-t_stage)/60:.1f}m")

# 2. Load dev-selected BEST config
best_file = WORK / "best_config.json"
BEST = json.loads(best_file.read_text(encoding="utf-8"))["BEST"] if best_file.exists() else {"model": "mbert", "tuning": "full"}
print(f"Starting Stage B (Kannada) + ROS_FRAC Ablation with BEST={BEST} | Current runs: {len(done_ids())}")

# 3. Kannada Stage B: FT (ce reused + wce, focal, ros) + Frozen (ce, wce, ros, smote)
kannada_stage_b = (
    [dict(lang="kannada", model=BEST["model"], tuning=BEST["tuning"], imb=imb, seed=s)
     for imb in FT_IMBALANCE for s in SEEDS]
    + [dict(lang="kannada", model=BEST["model"], tuning="frozen", imb=imb, seed=s)
       for imb in FROZEN_IMBALANCE for s in SEEDS]
)
execute(kannada_stage_b)

# 4. Kannada ROS_FRAC Ablation (0.25 and 1.0 for ros + frozen_smote)
kannada_ablation = []
for frac, tag in [(0.25, "f025"), (1.0, "f100")]:
    for s in SEEDS:
        kannada_ablation.append(dict(
            lang="kannada", model=BEST["model"], tuning=BEST["tuning"],
            imb=f"ros_{tag}", seed=s, ros_frac=frac
        ))
        kannada_ablation.append(dict(
            lang="kannada", model=BEST["model"], tuning="frozen",
            imb=f"frozen_smote_{tag}", seed=s, ros_frac=frac
        ))
execute(kannada_ablation)

# 5. Print summary table + milestone backup
results_df = pd.read_json(RESULTS, lines=True).drop_duplicates("run_id", keep="last")
sub_k = results_df[
    (results_df.lang == "kannada")
    & (results_df.model == BEST["model"])
    & (results_df.tuning.isin([BEST["tuning"], "frozen"]))
]

rows_k = []
for imb, g in sub_k.groupby("imb"):
    f1_str = (
        f"{100 * g.clean__macro_f1.mean():.2f} ± {100 * g.clean__macro_f1.std(ddof=1):.2f}"
        if len(g) > 1 else f"{100 * g.clean__macro_f1.mean():.2f}"
    )
    rows_k.append(dict(
        method=imb,
        n_seeds=len(g),
        macro_f1=f1_str,
        f1_mixed=round(100 * g.clean__f1_mixed_feelings.mean(), 2),
        f1_unknown=round(100 * g.clean__f1_unknown_state.mean(), 2),
        f1_not_in_lang=round(100 * g.clean__f1_not_in_language.mean(), 2),
        time_s=round(g.train_time_s.mean(), 1),
    ))

print("\n=== KANNADA STAGE-B & ABLATION SUMMARY ===")
print(pd.DataFrame(rows_k).sort_values("method").to_string(index=False))

save_portable_zip()
backup_now("cell6_kannada_stageB")

Starting Stage B (Kannada) + ROS_FRAC Ablation with BEST={'model': 'mbert', 'tuning': 'full'} | Current runs: 90
40 runs requested | 5 already done | 35 remaining
[1/35] kannada|mbert|full|wce|13 (lr=5e-05, ros_frac=0.5)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

    ep1 dev macro-F1 = 0.4222
    ep2 dev macro-F1 = 0.4593
    ep3 dev macro-F1 = 0.4793
  -> clean macro-F1 = 0.5029 | dev = 0.4793 | trainable 100.00% | 92s | Drive zip updated | elapsed 1.8m
[2/35] kannada|mbert|full|wce|21 (lr=5e-05, ros_frac=0.5)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

    ep1 dev macro-F1 = 0.4509
    ep2 dev macro-F1 = 0.4885
    ep3 dev macro-F1 = 0.4834
  -> clean macro-F1 = 0.4997 | dev = 0.4885 | trainable 100.00% | 83s | Drive zip updated | elapsed 3.4m
[3/35] kannada|mbert|full|wce|42 (lr=5e-05, ros_frac=0.5)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

... (43 outputs left out)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

    ep1 dev macro-F1 = 0.4865
    ep2 dev macro-F1 = 0.5102
    ep3 dev macro-F1 = 0.4964
  -> clean macro-F1 = 0.5043 | dev = 0.5102 | trainable 100.00% | 187s | Drive zip updated | elapsed 22.1m
[18/20] kannada|mbert|frozen|frozen_smote_f100|87 (lr=0.001, ros_frac=1.0)
  -> clean macro-F1 = 0.4812 | dev = 0.4834 | trainable 0.11% | 2s | Drive zip updated | elapsed 22.2m
[19/20] kannada|mbert|full|ros_f100|101 (lr=5e-05, ros_frac=1.0)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

    ep1 dev macro-F1 = 0.4685
    ep2 dev macro-F1 = 0.4780
    ep3 dev macro-F1 = 0.4837
  -> clean macro-F1 = 0.4980 | dev = 0.4837 | trainable 100.00% | 187s | Drive zip updated | elapsed 25.6m
[20/20] kannada|mbert|frozen|frozen_smote_f100|101 (lr=0.001, ros_frac=1.0)
  -> clean macro-F1 = 0.4526 | dev = 0.4464 | trainable 0.11% | 2s | Drive zip updated | elapsed 25.7m

=== KANNADA STAGE-B & ABLATION SUMMARY ===
           method  n_seeds     macro_f1  f1_mixed  f1_unknown  f1_not_in_lang  time_s
               ce        5 49.81 ± 1.24     12.89       39.46           62.30    82.7
            focal        5 49.79 ± 2.70     15.59       36.76           62.62    83.8
        frozen_ce        5 46.88 ± 1.05     14.83       34.94           58.22     0.8
       frozen_ros        5 47.87 ± 0.93     20.21       38.21           57.10     1.0
     frozen_smote        5 47.91 ± 1.82     21.33       36.47           60.29     1.0
frozen_smote_f025        5 47.83 ± 0.65     16.93       36.32   

In [ ]:
# 1. Re-enable per-run My Drive/dcm_bench_LATEST.zip auto-save (Cell 2's base execute() doesn't include it)
for sub in ["preds", "emb", "tables", "figs"]:
    (WORK / sub).mkdir(parents=True, exist_ok=True)

def save_portable_zip():
    tmp_zip = Path("/content/_tmp_dcm_latest.zip")
    with zipfile.ZipFile(tmp_zip, "w", compression=zipfile.ZIP_DEFLATED) as zf:
        for fname in ["results.jsonl", "results_pre_pooler_fix_backup.jsonl", "lr_config.json", "PLAN.md", "best_config.json"]:
            fp = WORK / fname
            if fp.exists():
                zf.write(fp, arcname=f"dcm_bench/main/{fname}")
        for folder in ["preds", "tables", "figs"]:
            fdir = WORK / folder
            if fdir.exists():
                for fp in fdir.rglob("*"):
                    if fp.is_file():
                        zf.write(fp, arcname=f"dcm_bench/{fp.relative_to(BASE_DIR)}")
        if DATA_DIR.exists():
            for fp in DATA_DIR.rglob("*"):
                if fp.is_file() and fp.suffix.lower() in (".csv", ".tsv", ".txt"):
                    zf.write(fp, arcname=f"dcm_bench/{fp.relative_to(BASE_DIR)}")
    shutil.copy2(tmp_zip, STORAGE_ROOT / "dcm_bench_LATEST.zip")
    tmp_zip.unlink(missing_ok=True)

def execute(runs):
    done = done_ids()
    todo = [r for r in runs if run_id(r) not in done]
    print(f"{len(runs)} runs requested | {len(runs) - len(todo)} already done | {len(todo)} remaining")
    t_stage = time.time()
    for n, r in enumerate(todo, 1):
        rid = run_id(r)
        frac = r.get("ros_frac", ROS_FRAC)
        lr_val = LR["head"] if r["imb"].startswith("frozen") else lr_for(r["tuning"])
        print(f"[{n}/{len(todo)}] {rid} (lr={lr_val}, ros_frac={frac})")
        try:
            if r["imb"].startswith("frozen"):
                info, sets, preds = train_frozen(r["lang"], r["model"], r["imb"], r["seed"], ros_frac=frac)
            else:
                info, sets, preds = train_finetune(r["lang"], r["model"], r["tuning"], r["imb"], r["seed"], ros_frac=frac)
        except torch.cuda.OutOfMemoryError:
            print("  OOM encountered — skipping run."); gc.collect(); torch.cuda.empty_cache(); continue

        row = dict(run_id=rid, lang=r["lang"], model=r["model"], tuning=r["tuning"], imb=r["imb"], seed=r["seed"], **info)
        for name, m in sets.items():
            row[f"{name}__macro_f1"]    = m["macro_f1"]
            row[f"{name}__acc"]         = m["acc"]
            row[f"{name}__weighted_f1"] = m["weighted_f1"]
        for c, f in zip(CLASSES, sets["clean"]["per_class_f1"]):
            row[f"clean__f1_{c}"] = f

        np.save(WORK / "preds" / (rid.replace("|", "__") + ".npy"), preds["clean"])
        if "noise_combined" in preds:
            np.save(WORK / "preds" / (rid.replace("|", "__") + "__noise_combined.npy"), preds["noise_combined"])

        with open(RESULTS, "a", encoding="utf-8") as fh:
            fh.write(json.dumps(row) + "\n")
            fh.flush()
            os.fsync(fh.fileno())

        save_portable_zip()
        gc.collect(); torch.cuda.empty_cache()
        print(f"  -> clean macro-F1 = {row['clean__macro_f1']:.4f} | dev = {info['dev_macro_f1']:.4f} | "
              f"trainable {info['trainable_pct']:.2f}% | {info['train_time_s']:.0f}s | Drive zip updated | elapsed {(time.time()-t_stage)/60:.1f}m")

# 2. Verify checkpoint & load BEST config
n_pre = len(done_ids())
print(f"Verified unique runs before Cell 7A: {n_pre} (expected >= 145)")
assert n_pre >= 145, f"Expected at least 145 completed runs, found {n_pre}."

best_file = WORK / "best_config.json"
BEST = json.loads(best_file.read_text(encoding="utf-8"))["BEST"] if best_file.exists() else {"model": "mbert", "tuning": "full"}
print(f"Starting Stage B Tamil Part 1 (wce + focal) with BEST={BEST}")

# 3. Run Tamil wce (5 seeds) + focal (5 seeds)
runs_tamil_7a = [
    dict(lang="tamil", model=BEST["model"], tuning=BEST["tuning"], imb=imb, seed=s)
    for imb in ["wce", "focal"]
    for s in SEEDS
]
execute(runs_tamil_7a)

# 4. Print Part-1 Tamil Stage-B summary + milestone backup
results_df = pd.read_json(RESULTS, lines=True).drop_duplicates("run_id", keep="last")
sub_t = results_df[
    (results_df.lang == "tamil")
    & (results_df.model == BEST["model"])
    & (results_df.tuning == BEST["tuning"])
    & (results_df.imb.isin(["ce", "wce", "focal"]))
]

rows_t = []
for imb, g in sub_t.groupby("imb"):
    f1_str = (
        f"{100 * g.clean__macro_f1.mean():.2f} ± {100 * g.clean__macro_f1.std(ddof=1):.2f}"
        if len(g) > 1 else f"{100 * g.clean__macro_f1.mean():.2f}"
    )
    rows_t.append(dict(
        method=imb,
        n_seeds=len(g),
        macro_f1=f1_str,
        f1_mixed=round(100 * g.clean__f1_mixed_feelings.mean(), 2),
        f1_unknown=round(100 * g.clean__f1_unknown_state.mean(), 2),
        f1_not_in_lang=round(100 * g.clean__f1_not_in_language.mean(), 2),
        time_min=round(g.train_time_s.mean() / 60, 2),
    ))

print("\n=== TAMIL STAGE-B PART 1 SUMMARY (ce / wce / focal) ===")
print(pd.DataFrame(rows_t).sort_values("method").to_string(index=False))

save_portable_zip()
backup_now("cell7a_tamil_stageB_part1")

Verified unique runs before Cell 7A: 145 (expected >= 145)
Starting Stage B Tamil Part 1 (wce + focal) with BEST={'model': 'mbert', 'tuning': 'full'}
10 runs requested | 0 already done | 10 remaining
[1/10] tamil|mbert|full|wce|13 (lr=5e-05, ros_frac=0.5)


config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/996k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.96M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  714MB            

... (17 outputs left out)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

    ep1 dev macro-F1 = 0.4096
    ep2 dev macro-F1 = 0.4598
    ep3 dev macro-F1 = 0.5018
  -> clean macro-F1 = 0.4862 | dev = 0.5018 | trainable 100.00% | 533s | Drive zip updated | elapsed 84.0m
[10/10] tamil|mbert|full|focal|101 (lr=5e-05, ros_frac=0.5)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

    ep1 dev macro-F1 = 0.4092
    ep2 dev macro-F1 = 0.4969
    ep3 dev macro-F1 = 0.4856
  -> clean macro-F1 = 0.4935 | dev = 0.4969 | trainable 100.00% | 529s | Drive zip updated | elapsed 93.3m

=== TAMIL STAGE-B PART 1 SUMMARY (ce / wce / focal) ===
method  n_seeds     macro_f1  f1_mixed  f1_unknown  f1_not_in_lang  time_min
    ce        5 49.15 ± 0.30     21.69       43.07           61.13      8.54
 focal        5 49.04 ± 0.65     20.86       42.57           60.86      8.81
   wce        5 49.41 ± 0.34     31.36       43.18           58.88      8.78
[BACKUP SAVED] /content/drive/MyDrive/dcm_bench/backups/dcm_backup_cell7a_tamil_stageB_part1.zip (0.38 MB) | Unique completed runs: 155


In [ ]:
# 1. Ensure per-run My Drive/dcm_bench_LATEST.zip auto-save is active
for sub in ["preds", "emb", "tables", "figs"]:
    (WORK / sub).mkdir(parents=True, exist_ok=True)

def save_portable_zip():
    tmp_zip = Path("/content/_tmp_dcm_latest.zip")
    with zipfile.ZipFile(tmp_zip, "w", compression=zipfile.ZIP_DEFLATED) as zf:
        for fname in ["results.jsonl", "results_pre_pooler_fix_backup.jsonl", "lr_config.json", "PLAN.md", "best_config.json"]:
            fp = WORK / fname
            if fp.exists():
                zf.write(fp, arcname=f"dcm_bench/main/{fname}")
        for folder in ["preds", "tables", "figs"]:
            fdir = WORK / folder
            if fdir.exists():
                for fp in fdir.rglob("*"):
                    if fp.is_file():
                        zf.write(fp, arcname=f"dcm_bench/{fp.relative_to(BASE_DIR)}")
        if DATA_DIR.exists():
            for fp in DATA_DIR.rglob("*"):
                if fp.is_file() and fp.suffix.lower() in (".csv", ".tsv", ".txt"):
                    zf.write(fp, arcname=f"dcm_bench/{fp.relative_to(BASE_DIR)}")
    shutil.copy2(tmp_zip, STORAGE_ROOT / "dcm_bench_LATEST.zip")
    tmp_zip.unlink(missing_ok=True)

def execute(runs):
    done = done_ids()
    todo = [r for r in runs if run_id(r) not in done]
    print(f"{len(runs)} runs requested | {len(runs) - len(todo)} already done | {len(todo)} remaining")
    t_stage = time.time()
    for n, r in enumerate(todo, 1):
        rid = run_id(r)
        frac = r.get("ros_frac", ROS_FRAC)
        lr_val = LR["head"] if r["imb"].startswith("frozen") else lr_for(r["tuning"])
        print(f"[{n}/{len(todo)}] {rid} (lr={lr_val}, ros_frac={frac})")
        try:
            if r["imb"].startswith("frozen"):
                info, sets, preds = train_frozen(r["lang"], r["model"], r["imb"], r["seed"], ros_frac=frac)
            else:
                info, sets, preds = train_finetune(r["lang"], r["model"], r["tuning"], r["imb"], r["seed"], ros_frac=frac)
        except torch.cuda.OutOfMemoryError:
            print("  OOM encountered — skipping run."); gc.collect(); torch.cuda.empty_cache(); continue

        row = dict(run_id=rid, lang=r["lang"], model=r["model"], tuning=r["tuning"], imb=r["imb"], seed=r["seed"], **info)
        for name, m in sets.items():
            row[f"{name}__macro_f1"]    = m["macro_f1"]
            row[f"{name}__acc"]         = m["acc"]
            row[f"{name}__weighted_f1"] = m["weighted_f1"]
        for c, f in zip(CLASSES, sets["clean"]["per_class_f1"]):
            row[f"clean__f1_{c}"] = f

        np.save(WORK / "preds" / (rid.replace("|", "__") + ".npy"), preds["clean"])
        if "noise_combined" in preds:
            np.save(WORK / "preds" / (rid.replace("|", "__") + "__noise_combined.npy"), preds["noise_combined"])

        with open(RESULTS, "a", encoding="utf-8") as fh:
            fh.write(json.dumps(row) + "\n")
            fh.flush()
            os.fsync(fh.fileno())

        save_portable_zip()
        gc.collect(); torch.cuda.empty_cache()
        print(f"  -> clean macro-F1 = {row['clean__macro_f1']:.4f} | dev = {info['dev_macro_f1']:.4f} | "
              f"trainable {info['trainable_pct']:.2f}% | {info['train_time_s']:.0f}s | Drive zip updated | elapsed {(time.time()-t_stage)/60:.1f}m")

# 2. Verify checkpoint & load BEST config
n_pre = len(done_ids())
print(f"Verified unique runs before Cell 7B: {n_pre} (expected >= 155)")
assert n_pre >= 155, f"Expected at least 155 completed runs, found {n_pre}."

best_file = WORK / "best_config.json"
BEST = json.loads(best_file.read_text(encoding="utf-8"))["BEST"] if best_file.exists() else {"model": "mbert", "tuning": "full"}
print(f"Starting Stage B Tamil Part 2 (ros + frozen_*) with BEST={BEST}")

# 3. Run Tamil ros (5 seeds) followed by all 4 frozen_* methods (20 runs)
runs_tamil_7b = (
    [dict(lang="tamil", model=BEST["model"], tuning=BEST["tuning"], imb="ros", seed=s)
     for s in SEEDS]
    + [dict(lang="tamil", model=BEST["model"], tuning="frozen", imb=imb, seed=s)
       for imb in FROZEN_IMBALANCE for s in SEEDS]
)
execute(runs_tamil_7b)

# 4. Print complete Tamil Stage-B summary (all 8 methods) + final training backup
results_df = pd.read_json(RESULTS, lines=True).drop_duplicates("run_id", keep="last")
sub_t = results_df[
    (results_df.lang == "tamil")
    & (results_df.model == BEST["model"])
    & (results_df.tuning.isin([BEST["tuning"], "frozen"]))
]

rows_t = []
for imb, g in sub_t.groupby("imb"):
    f1_str = (
        f"{100 * g.clean__macro_f1.mean():.2f} ± {100 * g.clean__macro_f1.std(ddof=1):.2f}"
        if len(g) > 1 else f"{100 * g.clean__macro_f1.mean():.2f}"
    )
    rows_t.append(dict(
        method=imb,
        n_seeds=len(g),
        macro_f1=f1_str,
        f1_mixed=round(100 * g.clean__f1_mixed_feelings.mean(), 2),
        f1_unknown=round(100 * g.clean__f1_unknown_state.mean(), 2),
        f1_not_in_lang=round(100 * g.clean__f1_not_in_language.mean(), 2),
        time_s=round(g.train_time_s.mean(), 1),
    ))

print("\n=== COMPLETE TAMIL STAGE-B SUMMARY (ALL 8 METHODS) ===")
print(pd.DataFrame(rows_t).sort_values("method").to_string(index=False))

save_portable_zip()
backup_now("cell7b_all_training_complete")

Verified unique runs before Cell 7B: 155 (expected >= 155)
Starting Stage B Tamil Part 2 (ros + frozen_*) with BEST={'model': 'mbert', 'tuning': 'full'}
25 runs requested | 0 already done | 25 remaining
[1/25] tamil|mbert|full|ros|13 (lr=5e-05, ros_frac=0.5)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

    ep1 dev macro-F1 = 0.4886
    ep2 dev macro-F1 = 0.4895
    ep3 dev macro-F1 = 0.4691
  -> clean macro-F1 = 0.4822 | dev = 0.4895 | trainable 100.00% | 901s | Drive zip updated | elapsed 15.5m
[2/25] tamil|mbert|full|ros|21 (lr=5e-05, ros_frac=0.5)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

    ep1 dev macro-F1 = 0.5015
    ep2 dev macro-F1 = 0.4935
    ep3 dev macro-F1 = 0.4825
  -> clean macro-F1 = 0.5034 | dev = 0.5015 | trainable 100.00% | 896s | Drive zip updated | elapsed 31.0m
[3/25] tamil|mbert|full|ros|42 (lr=5e-05, ros_frac=0.5)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

... (3 outputs left out)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

    ep1 dev macro-F1 = 0.4658
    ep2 dev macro-F1 = 0.4786
    ep3 dev macro-F1 = 0.4723
  -> clean macro-F1 = 0.4891 | dev = 0.4786 | trainable 100.00% | 897s | Drive zip updated | elapsed 77.3m
[6/25] tamil|mbert|frozen|frozen_ce|13 (lr=0.001, ros_frac=0.5)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

  -> clean macro-F1 = 0.4376 | dev = 0.4140 | trainable 0.11% | 4s | Drive zip updated | elapsed 78.1m
[7/25] tamil|mbert|frozen|frozen_ce|21 (lr=0.001, ros_frac=0.5)
  -> clean macro-F1 = 0.4206 | dev = 0.4082 | trainable 0.11% | 4s | Drive zip updated | elapsed 78.2m
[8/25] tamil|mbert|frozen|frozen_ce|42 (lr=0.001, ros_frac=0.5)
  -> clean macro-F1 = 0.4409 | dev = 0.4206 | trainable 0.11% | 4s | Drive zip updated | elapsed 78.4m
[9/25] tamil|mbert|frozen|frozen_ce|87 (lr=0.001, ros_frac=0.5)
  -> clean macro-F1 = 0.4231 | dev = 0.4085 | trainable 0.11% | 4s | Drive zip updated | elapsed 78.5m
[10/25] tamil|mbert|frozen|frozen_ce|101 (lr=0.001, ros_frac=0.5)
  -> clean macro-F1 = 0.4310 | dev = 0.4094 | trainable 0.11% | 5s | Drive zip updated | elapsed 78.7m
[11/25] tamil|mbert|frozen|frozen_wce|13 (lr=0.001, ros_frac=0.5)
  -> clean macro-F1 = 0.4301 | dev = 0.4175 | trainable 0.11% | 4s | Drive zip updated | elapsed 78.8m
[12/25] tamil|mbert|frozen|frozen_wce|21 (lr=0.001, ros_fr

In [ ]:
import os, json, re, time, shutil, zipfile, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from sklearn.metrics import f1_score, confusion_matrix

warnings.filterwarnings("ignore")

# ---------------- 1. Mount Drive & Verify 180-Run Checkpoint (CPU-Only) ----------------
from google.colab import drive
if not Path("/content/drive/MyDrive").exists():
    drive.mount("/content/drive")

STORAGE_ROOT = Path("/content/drive/MyDrive")
BASE_DIR     = STORAGE_ROOT / "dcm_bench"
WORK         = BASE_DIR / "main"
DATA_DIR     = BASE_DIR / "data"
RESULTS      = WORK / "results.jsonl"
TABLES_DIR   = WORK / "tables"
FIGS_DIR     = WORK / "figs"
PREDS_DIR    = WORK / "preds"

if not RESULTS.exists():
    zips = sorted(STORAGE_ROOT.glob("dcm_bench*.zip"), key=lambda p: p.stat().st_mtime, reverse=True)
    assert zips, "No dcm_bench*.zip found in My Drive."
    print(f"Unpacking {zips[0].name}...")
    with zipfile.ZipFile(zips[0], "r") as zf:
        members = [m for m in zf.namelist() if not m.startswith("__MACOSX")]
        target = STORAGE_ROOT if any(m.startswith("dcm_bench/") for m in members) else BASE_DIR
        for m in members:
            zf.extract(m, target)

TABLES_DIR.mkdir(parents=True, exist_ok=True)
FIGS_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_json(RESULTS, lines=True).drop_duplicates("run_id", keep="last").reset_index(drop=True)
print(f"Loaded {len(df)} unique runs from {RESULTS}")
assert len(df) == 180, f"Expected 180 runs, found {len(df)}."

BEST = (
    json.loads((WORK / "best_config.json").read_text(encoding="utf-8"))["BEST"]
    if (WORK / "best_config.json").exists()
    else {"model": "mbert", "tuning": "full"}
)
print("Locked BEST config:", BEST)

LANGS = ["tamil", "kannada"]
SEEDS = [13, 21, 42, 87, 101]
CLASSES = ["positive", "negative", "mixed_feelings", "unknown_state", "not_in_language"]
SHORT_CLS = ["Pos", "Neg", "Mixed", "Unk", "Not-Lang"]
C2I = {c: i for i, c in enumerate(CLASSES)}
FT_IMBALANCE     = ["ce", "wce", "focal", "ros"]
FROZEN_IMBALANCE = ["frozen_ce", "frozen_wce", "frozen_ros", "frozen_smote"]
ALL_STAGEB_IMB   = FT_IMBALANCE + FROZEN_IMBALANCE

# ---------------- 2. Reload Test Splits & Deterministic Combined Noise ----------------
def norm_label(l):
    l = re.sub(r"[\s\-]+", "_", str(l).strip().lower()).strip("_")
    if l.startswith("not_"): return "not_in_language"
    if l in ("mixed_feelings", "mixed", "mixed_feeling"): return "mixed_feelings"
    if l in ("unknown_state", "unknown"): return "unknown_state"
    return l

def read_local(path):
    with open(path, encoding="utf-8", errors="replace") as fh:
        raw_lines = [ln.strip() for ln in fh if ln.strip()]
    best_rows = []
    for sep in ("\t", ";", ","):
        rows = []
        for ln in raw_lines:
            parts = [p.strip() for p in ln.split(sep)]
            while parts and parts[-1] == "": parts.pop()
            if len(parts) < 2: continue
            lbl = norm_label(parts[-1])
            if lbl not in CLASSES: continue
            txt_parts = parts[1:-1] if (len(parts) >= 3 and re.fullmatch(r"\d+", parts[0])) else parts[:-1]
            txt = sep.join(txt_parts).strip()
            if txt: rows.append((txt, lbl))
        if len(rows) > len(best_rows): best_rows = rows
    return pd.DataFrame(best_rows, columns=["text", "label"])

def find_test_file(lang):
    cands = []
    for p in sorted(DATA_DIR.rglob("*")):
        n = p.name.lower()
        if "__macosx" in [x.lower() for x in p.parts] or n.startswith("."): continue
        if p.suffix.lower() not in (".tsv", ".csv", ".txt") or lang not in n or "offens" in n or "without_label" in n: continue
        if "test" in n:
            cands.append(p)
    cands.sort(key=lambda p: ("sentiment" not in p.name.lower(), p.name))
    return cands[0]

VOWELS = set("aeiou")
TRANSLIT_RULES = [
    ("aa", "a"), ("a", "aa"), ("ee", "i"), ("oo", "u"), ("u", "oo"),
    ("th", "t"), ("zh", "l"), ("sh", "s"), ("w", "v"), ("v", "w"),
    ("ai", "ei"), ("ph", "f"),
]
def _is_ascii_word(w): return w.isascii() and any(ch.isalpha() for ch in w)
def n_elongate(t, rng, p=0.15):
    out = []
    for ch in t:
        out.append(ch)
        if ch.lower() in VOWELS and ch.isascii() and rng.random() < p:
            out.append(ch * int(rng.integers(1, 3)))
    return "".join(out)
def n_typo(t, rng, p=0.05):
    chars, i, out = list(t), 0, []
    while i < len(chars):
        ch = chars[i]
        if ch.isascii() and ch.isalpha() and rng.random() < p:
            op = rng.integers(0, 3)
            if op == 0: pass
            elif op == 1 and i + 1 < len(chars): out.extend([chars[i + 1], ch]); i += 1
            else: out.extend([ch, ch])
        else: out.append(ch)
        i += 1
    return "".join(out)
def n_translit(t, rng, p_word=0.5):
    words = t.split(" ")
    for j, w in enumerate(words):
        if _is_ascii_word(w) and rng.random() < p_word:
            lw = w.lower()
            cand = [(a, b) for a, b in TRANSLIT_RULES if a in lw]
            if cand:
                a, b = cand[int(rng.integers(0, len(cand)))]
                words[j] = lw.replace(a, b)
    return " ".join(words)
def n_combined(t, rng): return n_typo(n_elongate(n_translit(t, rng), rng), rng)

TEST_DATA = {}
for lang in LANGS:
    te = read_local(find_test_file(lang))
    te["y"] = te["label"].map(C2I)
    rng = np.random.default_rng(0)
    te["noise_combined_text"] = [n_combined(t, rng) for t in te["text"].tolist()]
    TEST_DATA[lang] = te
    print(f"Loaded {lang} test set: {len(te)} rows")

# Helper: save CSV + LaTeX safely
def fmt_ms(s, scale=100.0):
    m = scale * s.mean()
    sd = scale * s.std(ddof=1) if len(s) > 1 else 0.0
    return f"{m:.2f} ± {sd:.2f}"

def save_table(tdf, stem):
    tdf.to_csv(TABLES_DIR / f"{stem}.csv", index=False)
    try:
        tex = tdf.to_latex(index=False, escape=True)
    except Exception:
        cols = list(tdf.columns)
        lines = [
            "\\begin{tabular}{" + "l" * len(cols) + "}",
            "\\hline",
            " & ".join(str(c).replace("_", "\\_") for c in cols) + " \\\\",
            "\\hline",
        ]
        for _, row in tdf.iterrows():
            lines.append(" & ".join(str(row[c]).replace("_", "\\_").replace("%", "\\%") for c in cols) + " \\\\")
        lines.extend(["\\hline", "\\end{tabular}"])
        tex = "\n".join(lines)
    (TABLES_DIR / f"{stem}.tex").write_text(tex, encoding="utf-8")

def load_preds(lang, model, tuning, imb, seed, suffix=""):
    fname = f"{lang}__{model}__{tuning}__{imb}__{seed}{suffix}.npy"
    fp = PREDS_DIR / fname
    return np.load(fp) if fp.exists() else None

def majority_vote(lang, model, tuning, imb, suffix=""):
    arrs = [load_preds(lang, model, tuning, imb, s, suffix) for s in SEEDS]
    arrs = [a for a in arrs if a is not None]
    if not arrs: return None
    mat = np.stack(arrs, axis=0)
    out = np.array([np.bincount(mat[:, i], minlength=len(CLASSES)).argmax() for i in range(mat.shape[1])])
    return out

# ---------------- 3. TABLE T1: Stage-A Main Benchmark ----------------
sa = df[df.imb == "ce"].copy()
t1_rows = []
for lang in LANGS:
    for m in ["mbert", "xlmr", "muril"]:
        for t in ["full", "lora_r32", "lora_r8"]:
            g = sa[(sa.lang == lang) & (sa.model == m) & (sa.tuning == t)]
            t1_rows.append(dict(
                lang=lang, model=m, tuning=t,
                dev_f1=fmt_ms(g.dev_macro_f1),
                clean_f1=fmt_ms(g.clean__macro_f1),
                weighted_f1=fmt_ms(g.clean__weighted_f1),
                acc=fmt_ms(g.clean__acc),
                trainable_pct=round(g.trainable_pct.mean(), 3),
                time_s=round(g.train_time_s.mean(), 1),
                vram_gb=round(g.peak_mem_gb.mean(), 2),
            ))
T1 = pd.DataFrame(t1_rows)
save_table(T1, "T1_stageA_main")
print("\n=== T1: STAGE-A MAIN BENCHMARK ===")
print(T1.to_string(index=False))

# ---------------- 4. TABLE T2: Stage-B Imbalance Comparison ----------------
sb = df[(df.model == BEST["model"]) & (df.tuning.isin([BEST["tuning"], "frozen"])) & (df.imb.isin(ALL_STAGEB_IMB))].copy()
t2_rows = []
for lang in LANGS:
    ce_mean = 100 * sb[(sb.lang == lang) & (sb.imb == "ce")].clean__macro_f1.mean()
    for imb in ALL_STAGEB_IMB:
        g = sb[(sb.lang == lang) & (sb.imb == imb)]
        m_f1 = 100 * g.clean__macro_f1.mean()
        t2_rows.append(dict(
            lang=lang, method=imb,
            macro_f1=fmt_ms(g.clean__macro_f1),
            delta_vs_ce=f"{m_f1 - ce_mean:+.2f}",
            f1_pos=round(100 * g.clean__f1_positive.mean(), 2),
            f1_neg=round(100 * g.clean__f1_negative.mean(), 2),
            f1_mixed=round(100 * g.clean__f1_mixed_feelings.mean(), 2),
            f1_unk=round(100 * g.clean__f1_unknown_state.mean(), 2),
            f1_notlang=round(100 * g.clean__f1_not_in_language.mean(), 2),
            time_s=round(g.train_time_s.mean(), 1),
        ))
T2 = pd.DataFrame(t2_rows)
save_table(T2, "T2_stageB_imbalance")
print("\n=== T2: STAGE-B IMBALANCE COMPARISON ===")
print(T2.to_string(index=False))

# ---------------- 5. TABLE T3: Orthographic Noise & Zero-Shot Cross-Lingual ----------------
t3_rows = []
for lang in LANGS:
    xl_col = "xl_kannada__macro_f1" if lang == "tamil" else "xl_tamil__macro_f1"
    for m in ["mbert", "xlmr", "muril"]:
        for t in ["full", "lora_r32", "lora_r8"]:
            g = sa[(sa.lang == lang) & (sa.model == m) & (sa.tuning == t)]
            cl = 100 * g.clean__macro_f1.mean()
            cb = 100 * g.noise_combined__macro_f1.mean()
            t3_rows.append(dict(
                lang=lang, model=m, tuning=t,
                clean=round(cl, 2),
                elong=round(100 * g.noise_elong__macro_f1.mean(), 2),
                typo=round(100 * g.noise_typo__macro_f1.mean(), 2),
                translit=round(100 * g.noise_translit__macro_f1.mean(), 2),
                combined=round(cb, 2),
                drop_comb=f"{cb - cl:+.2f}",
                zero_shot_xl=round(100 * g[xl_col].mean(), 2),
            ))
T3 = pd.DataFrame(t3_rows)
save_table(T3, "T3_robustness_transfer")
print("\n=== T3: ORTHOGRAPHIC NOISE & ZERO-SHOT CROSS-LINGUAL ===")
print(T3.to_string(index=False))

# ---------------- 6. TABLE T4: Kannada ROS_FRAC Ablation ----------------
ab_map = [
    ("ros", "ros_f025", 0.25), ("ros", "ros", 0.50), ("ros", "ros_f100", 1.00),
    ("frozen_smote", "frozen_smote_f025", 0.25), ("frozen_smote", "frozen_smote", 0.50), ("frozen_smote", "frozen_smote_f100", 1.00),
]
t4_rows = []
for family, imb_key, frac in ab_map:
    g = df[(df.lang == "kannada") & (df.model == BEST["model"]) & (df.imb == imb_key)]
    t4_rows.append(dict(
        family=family, ros_frac=frac,
        macro_f1=fmt_ms(g.clean__macro_f1),
        f1_mixed=round(100 * g.clean__f1_mixed_feelings.mean(), 2),
        f1_unk=round(100 * g.clean__f1_unknown_state.mean(), 2),
        f1_notlang=round(100 * g.clean__f1_not_in_language.mean(), 2),
        time_s=round(g.train_time_s.mean(), 1),
    ))
T4 = pd.DataFrame(t4_rows)
save_table(T4, "T4_ros_ablation")
print("\n=== T4: KANNADA ROS_FRAC ABLATION ===")
print(T4.to_string(index=False))

# ---------------- 7. TABLE T5: Statistical Significance Tests ----------------
def paired_seed_test(g_a, g_b, col):
    a = g_a.set_index("seed").loc[SEEDS, col].values
    b = g_b.set_index("seed").loc[SEEDS, col].values
    diff = 100 * (a - b)
    if np.allclose(diff, 0):
        return 0.0, 1.0
    _, p = stats.ttest_rel(a, b)
    return float(diff.mean()), float(p)

def mcnemar_and_bootstrap(y_true, pred_a, pred_b, B=2000, rng_seed=42):
    if pred_a is None or pred_b is None:
        return np.nan, np.nan
    # Exact two-sided McNemar on correctness
    corr_a = (pred_a == y_true)
    corr_b = (pred_b == y_true)
    n10 = int(np.sum(corr_a & ~corr_b))
    n01 = int(np.sum(~corr_a & corr_b))
    p_mcn = float(stats.binomtest(min(n10, n01), n10 + n01, 0.5).pvalue) if (n10 + n01) > 0 else 1.0
    # Paired bootstrap on macro-F1
    rng = np.random.default_rng(rng_seed)
    n = len(y_true)
    diffs = np.empty(B)
    for i in range(B):
        idx = rng.integers(0, n, size=n)
        fa = f1_score(y_true[idx], pred_a[idx], average="macro", labels=list(range(len(CLASSES))), zero_division=0)
        fb = f1_score(y_true[idx], pred_b[idx], average="macro", labels=list(range(len(CLASSES))), zero_division=0)
        diffs[i] = fa - fb
    p_boot = float(2 * min(np.mean(diffs <= 0), np.mean(diffs >= 0)))
    return p_mcn, min(1.0, p_boot)

t5_rows = []
for lang in LANGS:
    y_true = TEST_DATA[lang]["y"].values
    # Stage-A comparisons vs BEST (mbert|full|ce)
    g_best = df[(df.lang == lang) & (df.model == BEST["model"]) & (df.tuning == BEST["tuning"]) & (df.imb == "ce")]
    pv_best = majority_vote(lang, BEST["model"], BEST["tuning"], "ce")
    for (m, t) in [("mbert", "lora_r32"), ("mbert", "lora_r8"), ("xlmr", "full"), ("xlmr", "lora_r8"), ("muril", "lora_r8")]:
        g_cmp = df[(df.lang == lang) & (df.model == m) & (df.tuning == t) & (df.imb == "ce")]
        d_f1, p_t = paired_seed_test(g_cmp, g_best, "clean__macro_f1")
        pv_cmp = majority_vote(lang, m, t, "ce")
        p_mcn, p_boot = mcnemar_and_bootstrap(y_true, pv_cmp, pv_best)
        t5_rows.append(dict(
            lang=lang, block="Stage-A", comparison=f"{m}/{t} vs {BEST['model']}/{BEST['tuning']}",
            metric="clean_macro_f1", delta_pts=round(d_f1, 2),
            seed_ttest_p=round(p_t, 4), ens_mcnemar_p=round(p_mcn, 4), ens_boot_p=round(p_boot, 4),
        ))
    # Stage-B comparisons vs ce
    for imb in ["wce", "focal", "ros", "frozen_ce", "frozen_ros", "frozen_smote"]:
        tun = "frozen" if imb.startswith("frozen") else BEST["tuning"]
        g_cmp = df[(df.lang == lang) & (df.model == BEST["model"]) & (df.tuning == tun) & (df.imb == imb)]
        pv_cmp = majority_vote(lang, BEST["model"], tun, imb)
        p_mcn, p_boot = mcnemar_and_bootstrap(y_true, pv_cmp, pv_best)
        for metric_col, m_label in [("clean__macro_f1", "clean_macro_f1"), ("clean__f1_mixed_feelings", "f1_mixed")]:
            d_f1, p_t = paired_seed_test(g_cmp, g_best, metric_col)
            t5_rows.append(dict(
                lang=lang, block="Stage-B", comparison=f"{imb} vs ce",
                metric=m_label, delta_pts=round(d_f1, 2),
                seed_ttest_p=round(p_t, 4),
                ens_mcnemar_p=round(p_mcn, 4) if m_label == "clean_macro_f1" else np.nan,
                ens_boot_p=round(p_boot, 4) if m_label == "clean_macro_f1" else np.nan,
            ))

T5 = pd.DataFrame(t5_rows)
save_table(T5, "T5_significance_tests")
print("\n=== T5: STATISTICAL SIGNIFICANCE TESTS (p < 0.05 HIGHLIGHTED IN CSV) ===")
print(T5.to_string(index=False))

# ---------------- 8. PUBLICATION FIGURES (300 DPI PNG + PDF) ----------------
plt.rcParams.update({"font.size": 10, "axes.spines.top": False, "axes.spines.right": False})

# F1: Efficiency Pareto (Macro-F1 vs Trainable Params %)
fig, axes = plt.subplots(1, 2, figsize=(11, 4.3), sharey=True)
colors = {"mbert": "#1f77b4", "xlmr": "#ff7f0e", "muril": "#2ca02c"}
markers = {"full": "o", "lora_r32": "s", "lora_r8": "^"}
for ax, lang in zip(axes, LANGS):
    sub = sa[sa.lang == lang].groupby(["model", "tuning"]).agg(
        f1=("clean__macro_f1", lambda x: 100 * x.mean()),
        sd=("clean__macro_f1", lambda x: 100 * x.std(ddof=1)),
        pct=("trainable_pct", "mean"),
    ).reset_index()
    for _, r in sub.iterrows():
        ax.errorbar(r.pct, r.f1, yerr=r.sd, fmt=markers[r.tuning], color=colors[r.model],
                    markersize=8, capsize=3, label=f"{r.model}-{r.tuning}")
    ax.set_xscale("log")
    ax.set_title(f"{lang.capitalize()} — Clean Macro-F1 vs. Trainable Params (%)")
    ax.set_xlabel("Trainable Parameters (% of total, log scale)")
    ax.grid(True, linestyle="--", alpha=0.35)
axes[0].set_ylabel("Clean Test Macro-F1 (%)")
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="lower center", ncol=5, frameon=False, bbox_to_anchor=(0.5, -0.05))
fig.tight_layout(rect=[0, 0.08, 1, 1])
for ext in ("png", "pdf"): fig.savefig(FIGS_DIR / f"F1_efficiency_pareto.{ext}", dpi=300, bbox_inches="tight")
plt.close(fig)

# F2: Stage-B Macro-F1 vs Minority Class (mixed_feelings) F1
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), sharey=True)
x = np.arange(len(ALL_STAGEB_IMB))
w = 0.38
for ax, lang in zip(axes, LANGS):
    sub = sb[sb.lang == lang].groupby("imb").agg(
        mf1=("clean__macro_f1", lambda s: 100 * s.mean()),
        mixed=("clean__f1_mixed_feelings", lambda s: 100 * s.mean()),
    ).reindex(ALL_STAGEB_IMB)
    ax.bar(x - w/2, sub["mf1"], width=w, label="Overall Macro-F1", color="#2b5c8f")
    ax.bar(x + w/2, sub["mixed"], width=w, label="Mixed-Feelings F1", color="#e07a5f")
    ax.set_xticks(x)
    ax.set_xticklabels(ALL_STAGEB_IMB, rotation=35, ha="right")
    ax.set_title(f"{lang.capitalize()} — Imbalance Methods ({BEST['model']})")
    ax.grid(axis="y", linestyle="--", alpha=0.35)
axes[0].set_ylabel("Test F1 Score (%)")
axes[0].legend(frameon=False)
fig.tight_layout()
for ext in ("png", "pdf"): fig.savefig(FIGS_DIR / f"F2_imbalance_per_class.{ext}", dpi=300, bbox_inches="tight")
plt.close(fig)

# F3: Orthographic Noise Degradation (mBERT & XLM-R)
noise_cols = ["clean__macro_f1", "noise_elong__macro_f1", "noise_typo__macro_f1", "noise_translit__macro_f1", "noise_combined__macro_f1"]
noise_lbls = ["Clean", "Elong", "Typo", "Translit", "Combined"]
fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.3), sharey=True)
for ax, lang in zip(axes, LANGS):
    for m, ls in [("mbert", "-"), ("xlmr", "--")]:
        for t, mk in [("full", "o"), ("lora_r32", "s"), ("lora_r8", "^")]:
            g = sa[(sa.lang == lang) & (sa.model == m) & (sa.tuning == t)]
            vals = [100 * g[c].mean() for c in noise_cols]
            ax.plot(noise_lbls, vals, linestyle=ls, marker=mk, label=f"{m}/{t}")
    ax.set_title(f"{lang.capitalize()} — Orthographic Noise Robustness")
    ax.grid(True, linestyle="--", alpha=0.35)
axes[0].set_ylabel("Macro-F1 (%)")
axes[1].legend(frameon=False, ncol=2, fontsize=8)
fig.tight_layout()
for ext in ("png", "pdf"): fig.savefig(FIGS_DIR / f"F3_noise_robustness.{ext}", dpi=300, bbox_inches="tight")
plt.close(fig)

# F4: Kannada ROS_FRAC Ablation Curve
fig, ax = plt.subplots(figsize=(6.5, 4.2))
fracs = [0.25, 0.50, 1.00]
for fam, col, mk in [("ros", "#1f77b4", "o"), ("frozen_smote", "#d62728", "s")]:
    sub = T4[T4.family == fam].sort_values("ros_frac")
    means = [float(s.split("±")[0]) for s in sub.macro_f1]
    sds   = [float(s.split("±")[1]) for s in sub.macro_f1]
    ax.errorbar(fracs, means, yerr=sds, marker=mk, color=col, capsize=4, linewidth=2, label=f"{fam} (Macro-F1)")
    ax.plot(fracs, sub.f1_mixed.values, marker=mk, linestyle="--", color=col, alpha=0.7, label=f"{fam} (Mixed F1)")
ax.set_xticks(fracs)
ax.set_xlabel("Oversampling Target Ratio (ROS_FRAC)")
ax.set_ylabel("Kannada Test F1 (%)")
ax.set_title("Kannada Oversampling Ratio Ablation (mBERT)")
ax.grid(True, linestyle="--", alpha=0.35)
ax.legend(frameon=False, fontsize=8.5)
fig.tight_layout()
for ext in ("png", "pdf"): fig.savefig(FIGS_DIR / f"F4_ros_ablation.{ext}", dpi=300, bbox_inches="tight")
plt.close(fig)

# F5: Confusion Matrices (CE vs WCE, 5-seed ensemble)
fig, axes = plt.subplots(2, 2, figsize=(10.5, 9))
for row_i, lang in enumerate(LANGS):
    y_true = TEST_DATA[lang]["y"].values
    for col_i, imb in enumerate(["ce", "wce"]):
        ax = axes[row_i, col_i]
        pred = majority_vote(lang, BEST["model"], BEST["tuning"], imb)
        cm = confusion_matrix(y_true, pred, labels=list(range(len(CLASSES))), normalize="true") * 100
        im = ax.imshow(cm, cmap="Blues", vmin=0, vmax=100)
        for i in range(len(CLASSES)):
            for j in range(len(CLASSES)):
                ax.text(j, i, f"{cm[i, j]:.1f}", ha="center", va="center",
                        color="white" if cm[i, j] > 50 else "black", fontsize=8.5)
        ax.set_xticks(range(len(CLASSES))); ax.set_xticklabels(SHORT_CLS, rotation=30, ha="right")
        ax.set_yticks(range(len(CLASSES))); ax.set_yticklabels(SHORT_CLS)
        ax.set_title(f"{lang.capitalize()} — {BEST['model']}/{BEST['tuning']}/{imb.upper()} (%)")
        ax.set_xlabel("Predicted"); ax.set_ylabel("True")
fig.tight_layout()
for ext in ("png", "pdf"): fig.savefig(FIGS_DIR / f"F5_confusion_matrices.{ext}", dpi=300, bbox_inches="tight")
plt.close(fig)
print("\nSaved 5 publication figures (PNG + PDF) to:", FIGS_DIR)

# ---------------- 9. QUALITATIVE ERROR ANALYSIS CSVs ----------------
for lang in LANGS:
    te = TEST_DATA[lang].copy()
    y_true = te["y"].values
    p_ce   = majority_vote(lang, BEST["model"], BEST["tuning"], "ce")
    p_wce  = majority_vote(lang, BEST["model"], BEST["tuning"], "wce")
    p_ce_n = majority_vote(lang, BEST["model"], BEST["tuning"], "ce", suffix="__noise_combined")

    te["pred_ce"]  = [CLASSES[i] for i in p_ce]
    te["pred_wce"] = [CLASSES[i] for i in p_wce]
    te["pred_ce_noise"] = [CLASSES[i] for i in p_ce_n] if p_ce_n is not None else ""

    rescued = te[(te.label == "mixed_feelings") & (te.pred_ce != "mixed_feelings") & (te.pred_wce == "mixed_feelings")].head(25)
    both_wrong = te[(te.label == "mixed_feelings") & (te.pred_ce != "mixed_feelings") & (te.pred_wce != "mixed_feelings")].head(25)
    noise_flips = te[(te.pred_ce == te.label) & (te.pred_ce_noise != te.label)].head(25) if p_ce_n is not None else te.iloc[0:0]

    err_df = pd.concat([
        rescued.assign(category="wce_rescued_mixed"),
        both_wrong.assign(category="both_missed_mixed"),
        noise_flips.assign(category="noise_flipped_clean_correct"),
    ], ignore_index=True)
    err_df[["category", "text", "noise_combined_text", "label", "pred_ce", "pred_wce", "pred_ce_noise"]].to_csv(
        TABLES_DIR / f"error_analysis_{lang}.csv", index=False
    )
    print(f"[{lang}] Error analysis saved: {len(rescued)} WCE-rescued mixed | {len(both_wrong)} both-missed mixed | {len(noise_flips)} noise flips")

# ---------------- 10. CREATE FINAL PAPER ARTIFACT BUNDLE ON DRIVE ----------------
final_zip = STORAGE_ROOT / "dcm_bench_FINAL_PAPER_BUNDLE.zip"
tmp_zip = Path("/content/_tmp_final_bundle.zip")
with zipfile.ZipFile(tmp_zip, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for fname in ["results.jsonl", "results_pre_pooler_fix_backup.jsonl", "lr_config.json", "PLAN.md", "best_config.json"]:
        fp = WORK / fname
        if fp.exists():
            zf.write(fp, arcname=f"dcm_bench/main/{fname}")
    for folder in ["tables", "figs", "preds"]:
        fdir = WORK / folder
        if fdir.exists():
            for fp in sorted(fdir.rglob("*")):
                if fp.is_file():
                    zf.write(fp, arcname=f"dcm_bench/{fp.relative_to(BASE_DIR)}")
shutil.copy2(tmp_zip, final_zip)
tmp_zip.unlink(missing_ok=True)
mb = final_zip.stat().st_size / (1024 * 1024)
print(f"\n[ALL DONE] Final paper bundle saved to: {final_zip} ({mb:.2f} MB)")